# Paper 4, fourth revision runs

Five parts. Each runs in its own runtime, and all five can run at the same time; set the switches in the next cell.

1. **Part A (L4).** (a) The open decision models on `d3_goemotions`, then on `d2_banking77` (`laya-en` is smoke-tested on `d3_goemotions` only: its English head refused every 150-option question and may refuse the 77 Banking77 intents, which then leaves its `d3_goemotions` answers intact). (b) The four untuned Qwen3.5 backbones, key scoring, on the same two conditions. (c) The Qwen3-14B comparator `comparator-open` on the same two conditions. (d) The two 9B description-scoring backbones on the six conditions of the third revision. (e) Optional: a Jev latency sample from the cloud, if the Colab secret `TYPESAFE_API_KEY` exists.
2. **Part B (L4).** DeBERTa-v3-large fine-tuned per task: `baseline-deberta-large-clinc`, `-goemo`, `-banking`, `-d3` and `-d1`.
3. **Part C1 (A100 80GB).** Gemma-4-31B-it as a verbalized-JSON and an option-key-likelihood comparator on every condition Qwen3.6-27B covers.
4. **Part C2 (A100 80GB).** Mistral-Small-24B-Instruct-2501 in the same two modes on the same conditions.
5. **Part C3 (A100 80GB).** Qwen3.6-27B with thinking on (`comparator-open2-think`), then the Qwen3.6-27B pair on `d3_goemotions`, `d2_banking77` and the stress conditions.

All parts write to the same Drive folder. The last cell of whichever runtime finishes last lists and archives every part's files found there.

Before Run all, upload `revision4_upload.zip` to `My Drive/Jev/paper4/colab/`. The notebook unpacks it and checks every file against its SHA-256. Answers are written to `My Drive/Jev/paper4/colab/answers_revision4/` (the trained baselines write each condition at once), so a disconnect loses little: run all again and finished requests are skipped. A failing condition or model never stops the others. The last cell lists every planned file and writes `answers_revision4.zip` next to the folder for hand-back.

In [ ]:
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime

# ---- switches: one part per runtime (A and B on L4 runtimes, C1, C2 and C3 on A100 80GB)
RUN_PART_A = True
RUN_PART_B = False
RUN_PART_C1 = False          # Gemma-4-31B-it pair
RUN_PART_C2 = False          # Mistral-Small-24B pair
RUN_PART_C3 = False          # Qwen3.6-27B thinking, then the Qwen3.6-27B pair
# part A
RUN_DECISION_MODELS = True   # (a) seven open decision models
RUN_BACKBONE_KEY = True      # (b) the four Qwen3.5 backbones, key scoring
RUN_COMPARATOR_OPEN = True   # (c) comparator-open (Qwen3-14B-AWQ)
RUN_BACKBONE_DESC = True     # (d) the two 9B description-scoring backbones
RUN_JEV_LATENCY = True       # (e) 40 Jev requests; skipped without the TYPESAFE_API_KEY secret
# part B
B05_GROUPS = ['clinc', 'goemo', 'banking', 'd3', 'd1']   # run order
# part C3
RUN_THINK = True
RUN_QWEN36 = True
QWEN36_SMOKE = False         # the Qwen3.6 pair ran twice before with this env and pins
FREE_DISK_AFTER_DONE = True  # remove a comparator's downloaded weights once both its tags are done
HF_TOKEN_FROM_SECRETS = False  # True reads HF_TOKEN from the Colab secrets (faster downloads)
# compute units per hour, as Colab shows them in the Resources panel (assumptions until checked)
UNITS_PER_HOUR = {'L4': 4.8, 'A100': 11.8}

LOCAL_ROOT = '/content/p4'
SHARED_LOCAL = f'{LOCAL_ROOT}/shared'
SRC_DIR = f'{SHARED_LOCAL}/src'
INPUTS_DIR = f'{SHARED_LOCAL}/colab/inputs'
B05_DATA_DIR = f'{SHARED_LOCAL}/data/b05'
os.environ['HF_HOME'] = '/content/hf'
os.environ['USE_TF'] = '0'
if HF_TOKEN_FROM_SECRETS:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

DRIVE_ROOT = '/content/drive/MyDrive/Jev/paper4/colab'
UPLOAD_ZIP = f'{DRIVE_ROOT}/revision4_upload.zip'
ANSWERS_DIR = f'{DRIVE_ROOT}/answers_revision4'
os.makedirs(ANSWERS_DIR, exist_ok=True)
# one run log per combination of parts, so two runtimes never write the same file;
# the status cell merges them into run_log_revision4.json
RUN_PART_C = RUN_PART_C1 or RUN_PART_C2 or RUN_PART_C3
# (part C3 split over two runtimes: C3think and C3qwen)
_c3 = 'C3' + ('' if RUN_THINK == RUN_QWEN36 else ('think' if RUN_THINK else 'qwen'))
LABEL = '_'.join(p for p, on in (('A', RUN_PART_A), ('B', RUN_PART_B), ('C1', RUN_PART_C1),
                                 ('C2', RUN_PART_C2), (_c3, RUN_PART_C3)) if on)
RUN_LOG_PATH = f'{ANSWERS_DIR}/run_log_revision4_part{LABEL or "none"}.json'
run_log = json.load(open(RUN_LOG_PATH)) if os.path.exists(RUN_LOG_PATH) else {}
run_log.setdefault('runtimes', []).append({'start': datetime.datetime.utcnow().isoformat() + 'Z',
                                           'parts': LABEL, 'b05_groups': B05_GROUPS,
                                           'think': RUN_THINK, 'qwen36': RUN_QWEN36})
run_log['units_per_hour_assumed'] = UNITS_PER_HOUR
print('answers on Drive:', ANSWERS_DIR, '| parts:', LABEL or 'none (status only)')

In [ ]:
import subprocess, shutil
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
run_log['runtimes'][-1]['gpu'] = gpu_info
PART_C_GPU_OK = 'A100' in gpu_info or 'H100' in gpu_info
print(gpu_info)
print(f"free disk under /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
if RUN_PART_C and not PART_C_GPU_OK:
    print('WARNING: parts C1, C2 and C3 need an A100 (80 GB planned); their cells will skip on this GPU.')
elif RUN_PART_C and int(''.join(ch for ch in gpu_info.split(',')[-1] if ch.isdigit()) or 0) < 70000:
    print('note: parts C1, C2 and C3 were planned for an 80 GB A100; on 40 GB less memory is left for the '
          'prompt cache and the likelihood runs take longer.')
if (RUN_PART_A or RUN_PART_B) and 'L4' not in gpu_info:
    print('note: parts A and B were planned for an L4; latencies will not match the earlier runs.')

## Unpack the upload and verify every file

In [ ]:
import hashlib, zipfile

EXPECTED = {
 "d1_calib": {
  "decisions": 1500,
  "file": "d1_calib.jsonl",
  "requests": 300,
  "sha256": "e361865af906b14d76ef284a5bc59f66077e05b992783c435d649bbe74b3b93c"
 },
 "d1_neutral": {
  "decisions": 2000,
  "file": "d1_neutral.jsonl",
  "requests": 400,
  "sha256": "013fd3993a5d3e4e77cd267161d6333a202a35ca0daa2342abaf945f8548b591"
 },
 "d2_banking77": {
  "decisions": 616,
  "file": "d2_banking77.jsonl",
  "requests": 616,
  "sha256": "8540748e94e90142678c178b13901d125444f338e37cb8fa14569a547feab78a"
 },
 "d2_k150": {
  "decisions": 800,
  "file": "d2_k150.jsonl",
  "requests": 800,
  "sha256": "b41706d09f2bd228463d50b5e1cd81d45c7da2fd3abaa5ce7fd869aaabd9a1ec"
 },
 "d2_k150_oos": {
  "decisions": 800,
  "file": "d2_k150_oos.jsonl",
  "requests": 800,
  "sha256": "7d991da0c4f71063e92ecac3c21cf2d7bcc5ee5ffbecdae40a5797e9b59ba73a"
 },
 "d2_k20": {
  "decisions": 600,
  "file": "d2_k20.jsonl",
  "requests": 600,
  "sha256": "d35c78e0a0541bc911268db6b56f4d644335ee76b1c907d4f9857a56b1041c98"
 },
 "d2_k5": {
  "decisions": 600,
  "file": "d2_k5.jsonl",
  "requests": 600,
  "sha256": "e0dbee0ded6b07ab9fd8b372287561c640f39ac436bb5782bcd24f6bea71d8a2"
 },
 "d2_k50": {
  "decisions": 600,
  "file": "d2_k50.jsonl",
  "requests": 600,
  "sha256": "7fd1cc8f33e947e01ca7136df96164ef09b7d5ea0ac65572a948df6ef9dfbb08"
 },
 "d3_conv_go_awry": {
  "decisions": 500,
  "file": "d3_conv_go_awry.jsonl",
  "requests": 500,
  "sha256": "de3ef7269832338cee4c271d23b9a4a18ea9919d55937bb1eb26612ee29fe712"
 },
 "d3_emotion": {
  "decisions": 498,
  "file": "d3_emotion.jsonl",
  "requests": 498,
  "sha256": "ccd5a620b9269d577a0f9672351b0eaa1b2f03fd0527eadaf9884e3078cc9f9b"
 },
 "d3_goemotions": {
  "decisions": 492,
  "file": "d3_goemotions.jsonl",
  "requests": 492,
  "sha256": "3bb9ac4ad9d7b99c8002ca3e50b81c2c9eff2ab90335f668577e30d8fc924da6"
 },
 "d3_wiki_corpus": {
  "decisions": 500,
  "file": "d3_wiki_corpus.jsonl",
  "requests": 500,
  "sha256": "8b24c2f580b834d61971029eb4b8af94fc58d186b4673f3fe2a57b33557436d3"
 },
 "d3_wiki_politeness": {
  "decisions": 498,
  "file": "d3_wiki_politeness.jsonl",
  "requests": 498,
  "sha256": "a98b22a9efbac95da3dca46eaa40f5ac0b3f024527aa823f8e73446f74596e68"
 },
 "e2_d1_k01": {
  "decisions": 600,
  "file": "e2_d1_k01.jsonl",
  "requests": 400,
  "sha256": "9ea1cf5240cb73e4b795a10663afb687fc65228ef9cce5690ecb9d212fa0bf98"
 },
 "e2_d1_kny": {
  "decisions": 600,
  "file": "e2_d1_kny.jsonl",
  "requests": 400,
  "sha256": "73b0ebcfcfb734ac73f4c1f1e501b1f964344907d5114584c17b3878a79aa1f6"
 },
 "e2_d1_krand": {
  "decisions": 600,
  "file": "e2_d1_krand.jsonl",
  "requests": 400,
  "sha256": "4ae671a3e091c14e78dca097ea7cb590690ae898d7d54d5e1423f51af7401803"
 },
 "e2_d1_kswap": {
  "decisions": 600,
  "file": "e2_d1_kswap.jsonl",
  "requests": 400,
  "sha256": "106ab580a8274a88c9e504c688c8d0d1da232b3846f4d6ce55de6e55085c3ca8"
 },
 "e2_d3_conv_go_awry_k01": {
  "decisions": 500,
  "file": "e2_d3_conv_go_awry_k01.jsonl",
  "requests": 500,
  "sha256": "cf178f09d2a23b949b963cf527b1a8c55af6285196609edb3c79adbf8a02b728"
 },
 "e2_d3_conv_go_awry_kny": {
  "decisions": 500,
  "file": "e2_d3_conv_go_awry_kny.jsonl",
  "requests": 500,
  "sha256": "f3615a1592e3aad413ee0fa58d00d63635e9b2439d89aad2ed4755afd66c8c4f"
 },
 "e2_d3_conv_go_awry_krand": {
  "decisions": 500,
  "file": "e2_d3_conv_go_awry_krand.jsonl",
  "requests": 500,
  "sha256": "287bf4c6ba1e0c5af8f782869fa2bd5f01ba3a47de5613678801209b44e0802e"
 },
 "e2_d3_conv_go_awry_kswap": {
  "decisions": 500,
  "file": "e2_d3_conv_go_awry_kswap.jsonl",
  "requests": 500,
  "sha256": "329c2641487bf27954e5150fc05ec5652a7bcccd609402c6b4ee15a19b6e74a0"
 },
 "e2_d3_wiki_corpus_k01": {
  "decisions": 500,
  "file": "e2_d3_wiki_corpus_k01.jsonl",
  "requests": 500,
  "sha256": "a778a5c5904bb1234aaf256d593275c29ebfdc6460e25d0ad974b68dcb0d9bda"
 },
 "e2_d3_wiki_corpus_kny": {
  "decisions": 500,
  "file": "e2_d3_wiki_corpus_kny.jsonl",
  "requests": 500,
  "sha256": "01147e9e891551699ea9bcda03e890758e11aa16e7f90ea6a72b4409ae81eaa6"
 },
 "e2_d3_wiki_corpus_krand": {
  "decisions": 500,
  "file": "e2_d3_wiki_corpus_krand.jsonl",
  "requests": 500,
  "sha256": "f8ea689c90b1fe52e9748b93371fa28531c2bb5d1ec9f76a050601b2d063e9e6"
 },
 "e2_d3_wiki_corpus_kswap": {
  "decisions": 500,
  "file": "e2_d3_wiki_corpus_kswap.jsonl",
  "requests": 500,
  "sha256": "0c3248232c3fcb39aa3eb5ba32d91a09d2d587f6b9b43ae8a39ff6ff90656d22"
 }
}
SRC_SHA256 = {
 "adapters.py": "36b6c5340fe91b6204d067ab050d6094c91c11a5265e3e8b13b3270409f8d73b",
 "b05_strong_classifiers.py": "61076975aa7c842e4a5227c12c00290f961717a0979d4aa1432e8a5aa0929f2d",
 "common.py": "5fc5d29c76ac4b1459161ad74b199c865fbfe5cf0d49767ab7b5b985e0a3aaca",
 "harness.py": "2be4b3216600dc518126c62b1299cb6abe3e6626fb496fc8e4862d1f276af162",
 "metrics.py": "97f0ee7eeec9601d1ce0647cb41a145d5bab8c9afb38b352ecb9ac24e6a5fd83"
}
B05_DATA_SHA256 = {
 "b05_meta.json": "b483ea8d992d5a1d102a459d67456d5752574705c1745794cdf3dc2f20091418",
 "banking77.jsonl": "9dd1c10f79f68f2458125e18d3808d23a74aa7b2aa58cdfab35faae4a881f38c",
 "clinc.jsonl": "12a7693e8f6d82cf100b2ea4fc5149e1b2a50bdd51f5cfb74f806d2ab45948e9",
 "conv_go_awry.jsonl": "0199357d40e55e5f2df5c44fb73ce98b38293bec70849c3da1a73ab676ffc695",
 "d1.jsonl": "6951d387a490bddda49d5d69c7a388abad23c495a469fdd99f558025db0b91e1",
 "emotion.jsonl": "62b6de6b198cf4771f430dbb708176608b8f2ba35f7a2e340e950968452f4ac6",
 "goemotions.jsonl": "58369e0dd6d4bcd24a90be952fddc2a88f7d89abed07aeec0f79c7f239e8755a",
 "wiki_corpus.jsonl": "3fa9d5c2a9d1c30b2dac86d4bc6e8908208729f478d172c1f5b9326a0cb82984",
 "wiki_politeness.jsonl": "5b98e112d0b7ed8c5ffd5d63590f278ee4d459c2c5877c814ba866aae56294d9"
}
ZIP_SHA256 = '5257d1a6a47305a6dfe9596df3bef2bb3ecc9a51089d7bbc0cb71280a961e686'

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not os.path.exists(UPLOAD_ZIP):
    raise SystemExit(f'{UPLOAD_ZIP} not found: upload revision4_upload.zip to that Drive folder first.')
got = _sha256(UPLOAD_ZIP)
if got != ZIP_SHA256:
    print(f'note: the zip differs from the one this notebook was built with ({got}); '
          'the per-file check below decides')
shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
with zipfile.ZipFile(UPLOAD_ZIP) as z:
    z.extractall(LOCAL_ROOT)
bad = []
for cond, meta in EXPECTED.items():
    p = f'{INPUTS_DIR}/' + meta['file']
    if not os.path.exists(p) or _sha256(p) != meta['sha256']:
        bad.append(cond)
for f, h in SRC_SHA256.items():
    if _sha256(f'{SRC_DIR}/{f}') != h:
        bad.append(f)
for f, h in B05_DATA_SHA256.items():
    if _sha256(f'{B05_DATA_DIR}/{f}') != h:
        bad.append(f)
if bad:
    raise SystemExit(f'these files do not match the build: {bad} -- stopping.')
print(f'{len(EXPECTED)} input files, {len(SRC_SHA256)} source files and '
      f'{len(B05_DATA_SHA256)} training data files verified.')

## Revisions, plan and helpers

Every checkpoint is pinned through `P4_REVISIONS`, which `adapters.py` reads: the decision models and Qwen3-14B-AWQ to the commits of their original answers, the backbones and Qwen3.6-27B to the commits of the second revision run, Gemma-4-31B-it to `842da379`, Mistral-Small-24B-Instruct-2501 to `9527884be6`. `b05_strong_classifiers.py` pins DeBERTa-v3-large to `64a8c8ea` itself.

In [ ]:
REVISIONS = {
 "Mapika/decider-2b": "d61c1c16089572df5d180329b9fea4997a90090c",
 "Qwen/Qwen3-14B-AWQ": "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
 "Qwen/Qwen3.5-0.8B-Base": "dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68",
 "Qwen/Qwen3.5-2B-Base": "b1485b2fa6dfa1287294f269f5fb618e03d52d7c",
 "Qwen/Qwen3.5-9B": "c202236235762e1c871ad0ccb60c8ee5ba337b9a",
 "Qwen/Qwen3.5-9B-Base": "68c46c4b3498877f3ef123c856ecfde50c39f404",
 "Qwen/Qwen3.6-27B-FP8": "e89b16ebf1988b3d6befa7de50abc2d76f26eb09",
 "bespokelabs/Bespoke-Nimble-9B": "bd792f44ec8e265be861bfcdf4e05967ffe0e858",
 "convaiinnovations/laya": "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851",
 "cyankiwi/Qwen3.6-27B-AWQ-INT4": "e5cc0400fb2403c437c2c40a7c52fb5ae93fda18",
 "flock-io/this-that-model-1.0": "3d927195c4f9845efe66c5715883a7a0f42b1239",
 "google/gemma-4-31B-it": "842da3794eaa0b77d5f08bae87a17459d91ff475",
 "jaredpalmer/kev-0.8b": "9a45d25eb2ab761841196625383fa1dff0e56c1e",
 "jaredpalmer/kev-9b": "2629c06a5aeb0feb3b9783bafed17ed8f39ecf5c",
 "microsoft/deberta-v3-large": "64a8c8eab3e352a784c658aef62be1662607476f",
 "mistralai/Mistral-Small-24B-Instruct-2501": "9527884be6e5616bdd54de542f9ae13384489724"
}
TAG_REPO = {
 "laya-en": "convaiinnovations/laya",
 "laya-ml": "convaiinnovations/laya",
 "kev-0.8b": "jaredpalmer/kev-0.8b",
 "decider-2b": "Mapika/decider-2b",
 "this-that-1.0": "flock-io/this-that-model-1.0",
 "kev-9b": "jaredpalmer/kev-9b",
 "nimble-9b": "bespokelabs/Bespoke-Nimble-9B",
 "comparator-open": "Qwen/Qwen3-14B-AWQ",
 "backbone-desc-qwen35-9b-base": "Qwen/Qwen3.5-9B-Base",
 "backbone-desc-qwen35-9b": "Qwen/Qwen3.5-9B",
 "baseline-deberta-large-clinc": "microsoft/deberta-v3-large",
 "baseline-deberta-large-goemo": "microsoft/deberta-v3-large",
 "baseline-deberta-large-banking": "microsoft/deberta-v3-large",
 "baseline-deberta-large-d3": "microsoft/deberta-v3-large",
 "baseline-deberta-large-d1": "microsoft/deberta-v3-large",
 "comparator-gemma": "google/gemma-4-31B-it",
 "comparator-gemma-ll": "google/gemma-4-31B-it",
 "comparator-mistral": "mistralai/Mistral-Small-24B-Instruct-2501",
 "comparator-mistral-ll": "mistralai/Mistral-Small-24B-Instruct-2501",
 "comparator-open2": "Qwen/Qwen3.6-27B-FP8",
 "comparator-open2-ll": "Qwen/Qwen3.6-27B-FP8",
 "backbone-qwen35-0.8b-base": "Qwen/Qwen3.5-0.8B-Base",
 "backbone-qwen35-2b-base": "Qwen/Qwen3.5-2B-Base",
 "backbone-qwen35-9b-base": "Qwen/Qwen3.5-9B-Base",
 "backbone-qwen35-9b": "Qwen/Qwen3.5-9B",
 "comparator-open2-think": "Qwen/Qwen3.6-27B-FP8",
 "comparator-gemma-b1": "google/gemma-4-31B-it",
 "comparator-gemma-ll-b1": "google/gemma-4-31B-it",
 "comparator-mistral-b1": "mistralai/Mistral-Small-24B-Instruct-2501",
 "comparator-mistral-ll-b1": "mistralai/Mistral-Small-24B-Instruct-2501",
 "comparator-open-b1": "Qwen/Qwen3-14B-AWQ",
 "jev-1.13.0": "jev-1.13.0"
}

import subprocess, shutil, traceback, gc

def answer_counts(path):
    """(answered lines, error lines) of one answer file; (0, 0) when it does not exist."""
    ok = err = 0
    if os.path.exists(path):
        with open(path, encoding='utf-8') as fh:
            for l in fh:
                try:
                    j = json.loads(l)
                except json.JSONDecodeError:
                    continue
                if j.get('error'):
                    err += 1
                else:
                    ok += 1
    return ok, err

def n_ok(tag, cond):
    return answer_counts(f'{ANSWERS_DIR}/{tag}/{cond}__rep1.jsonl')[0]

def incomplete(tag, conds):
    return [c for c in conds if n_ok(tag, c) < EXPECTED[c]['requests']]

def unfinished(tag, conds):
    """Conditions with requests that have no line at all (answered or error), after a crash."""
    return [c for c in conds
            if sum(answer_counts(f'{ANSWERS_DIR}/{tag}/{c}__rep1.jsonl')) < EXPECTED[c]['requests']]

def smoke_counts(smoke_dir):
    """(answered lines, error lines, output tokens per answered question) of the smoke files."""
    ok = err = 0
    outs = []
    for root, _dirs, files in os.walk(smoke_dir):
        for fn in sorted(files):
            if not fn.endswith('.jsonl'):
                continue
            with open(os.path.join(root, fn), encoding='utf-8') as fh:
                for l in fh:
                    try:
                        j = json.loads(l)
                    except json.JSONDecodeError:
                        continue
                    if j.get('error'):
                        err += 1
                        continue
                    ok += 1
                    for a in (j.get('answers') or {}).values():
                        u = (a.get('raw') or {}).get('usage_out')
                        if u is not None:
                            outs.append(u)
    return ok, err, outs

def smoke_report(smoke_dir):
    """Error lines and the largest self-check difference over every smoke answer file.
    harness.py records a failed request, a failed self-check included, as an error line and
    exits 0, so the smoke gate reads the files instead of trusting the exit code."""
    errs, dmax = [], None
    for root, _dirs, files in os.walk(smoke_dir):
        for fn in sorted(files):
            if not fn.endswith('.jsonl'):
                continue
            with open(os.path.join(root, fn), encoding='utf-8') as fh:
                for l in fh:
                    try:
                        j = json.loads(l)
                    except json.JSONDecodeError:
                        continue
                    if j.get('error'):
                        errs.append(f"{fn}: {j['error'][:200]}")
                        continue
                    for a in (j.get('answers') or {}).values():
                        d = (a.get('raw') or {}).get('check_max_abs_dprob')
                        if d is not None:
                            dmax = d if dmax is None else max(dmax, d)
    return errs, dmax

def mark(tag, ok, detail=''):
    """DONE_revision4 or ERROR_revision4.txt in the tag folder; an earlier error marker of this
    notebook is kept as ERROR_revision4_earlier.txt."""
    d = f'{ANSWERS_DIR}/{tag}'
    os.makedirs(d, exist_ok=True)
    if os.path.exists(f'{d}/ERROR_revision4.txt'):
        os.replace(f'{d}/ERROR_revision4.txt', f'{d}/ERROR_revision4_earlier.txt')
    name = 'DONE_revision4' if ok else 'ERROR_revision4.txt'
    with open(f'{d}/{name}', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n' + detail)
    print(f'=== {tag}: {"DONE" if ok else "ERROR"} {detail[:400]} ===', flush=True)

def is_done(tag):
    return os.path.exists(f'{ANSWERS_DIR}/{tag}/DONE_revision4')

def stream(cmd, log_path, env=None):
    """Run a command; every output line goes to the cell and to the log on Drive."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return p.wait()

def harness_env(env_name, answers_dir, extra=None):
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = answers_dir
    env['P4_REVISIONS'] = json.dumps(REVISIONS)
    # ninja and any other console script of the env must be on PATH (FlashInfer JIT, Triton)
    env['PATH'] = f'/content/envs/{env_name}/bin:' + env.get('PATH', '')
    env.update(extra or {})
    return env

def harness(env_name, tag, conds, log_path, answers_dir=None, limit=None, extra=None):
    """Conditions in one process (the model loads once), rep 1."""
    cmd = [f'/content/envs/{env_name}/bin/python', f'{SRC_DIR}/harness.py', '--model', tag,
           '--cond', ','.join(conds), '--rep', '1']
    if limit:
        cmd += ['--limit', str(limit)]
    t0 = time.time()
    rc = stream(cmd, log_path, harness_env(env_name, answers_dir or ANSWERS_DIR, extra))
    return rc, time.time() - t0

def save_run_log():
    tmp = RUN_LOG_PATH + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(run_log, fh, indent=2)
    os.replace(tmp, RUN_LOG_PATH)

def free_gb():
    return shutil.disk_usage('/content').free / 1e9

def gpu_memory():
    subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'])

print(json.dumps(REVISIONS, indent=1))

## Part A (a): open decision models on d3_goemotions and d2_banking77

Each cell builds (or reuses) its uv env, smoke-tests 3 requests of each condition (`laya-en`: of `d3_goemotions` only; any error line stops that model), prints a time estimate, then runs each condition and retries it once if it is incomplete. Output is streamed and appended to `answers_revision4/<tag>/run_log_<tag>.txt`. Look for `DONE_revision4` or `ERROR_revision4.txt` in each model folder.

### laya-en

In [ ]:
TAG = 'laya-en'
ENV_NAME = 'laya'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/laya --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/laya/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 laya==0.3.20 python-dotenv httpx numpy\n'
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### laya-ml

In [ ]:
TAG = 'laya-ml'
ENV_NAME = 'laya'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/laya --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/laya/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 laya==0.3.20 python-dotenv httpx numpy\n'
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### kev-0.8b

In [ ]:
TAG = 'kev-0.8b'
ENV_NAME = 'kev_decider_tt'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### decider-2b

In [ ]:
TAG = 'decider-2b'
ENV_NAME = 'kev_decider_tt'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### this-that-1.0

In [ ]:
TAG = 'this-that-1.0'
ENV_NAME = 'kev_decider_tt'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f' 'thisthat @ git+https://github.com/FLock-io/this-that-model@542d445efa5f68b14bfbd1f8ed25aacd8379d839'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### kev-9b

In [ ]:
TAG = 'kev-9b'
ENV_NAME = 'kev_decider_tt'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/kev_decider_tt --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 'transformers>=5.17,<6' 'peft>=0.21' 'accelerate>=1.15' huggingface_hub python-dotenv httpx pydantic numpy scipy 'kev @ git+https://github.com/jaredpalmer/kev@73504e51f6ce2ade19c7819d4a5f2d84363cd40f'\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/kev_decider_tt/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### nimble-9b

In [ ]:
TAG = 'nimble-9b'
ENV_NAME = 'nimble'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/nimble --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 sentencepiece==0.2.2 pillow==12.3.0 'huggingface_hub>=0.34' python-dotenv httpx numpy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/nimble/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']   # rep 1 each, most important first
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']

if not (RUN_PART_A and RUN_DECISION_MODELS):
    print(f'{TAG}: skipped (RUN_PART_A = {RUN_PART_A}, RUN_DECISION_MODELS = {RUN_DECISION_MODELS})')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    # a fresh smoke folder on every run of the cell, so the smoke test and its timing are real
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        print(f'=== {TAG}: smoke test, 3 requests of each of {SMOKE_CONDS} ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, SMOKE_CONDS, LOG, answers_dir=SMOKE_DIR, limit=3)
        errs, _ = smoke_report(SMOKE_DIR)
        if rc or errs:
            raise RuntimeError(f'smoke test failed (exit {rc}, {len(errs)} error lines, first: '
                               f'{errs[:1]}); see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        per = dt / (3 * len(SMOKE_CONDS))
        print(f'{TAG}: {per:.2f} s per request including the model load -> at most '
              f'{per * n_req / 60:.0f} min for {n_req} requests, '
              f'~{per * n_req / 3600 * UNITS_PER_HOUR["L4"]:.1f} compute units on an L4', flush=True)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_per_request_incl_load': per,
                                                 'requests': n_req}
        for cond in CONDS:
            for attempt in (1, 2):
                rc, dt = harness(ENV_NAME, TAG, [cond], LOG)
                ok, err = answer_counts(f'{ANSWERS_DIR}/{TAG}/{cond}__rep1.jsonl')
                want = EXPECTED[cond]['requests']
                print(f'[{TAG}] {cond} attempt {attempt}: exit {rc}, {dt / 60:.1f} min, '
                      f'{ok}/{want} answered, {err} error lines', flush=True)
                run_log['models'][TAG][f'{cond}_min_attempt{attempt}'] = dt / 60
                if ok >= want:
                    break
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


## Part A (b): the untuned backbones, key scoring, on d3_goemotions and d2_banking77

The four `backbone-qwen35-*` tags of the second revision (`BackboneOptScoreBackend`), in its `backbone` env with its pins. Each cell smoke-tests 2 requests of each condition with the cached-versus-full-sequence self-check on (the key-scoring check passed on `d2_k150` for all four in the second revision), then runs both conditions in one process. Switch `RUN_BACKBONE_KEY`.

### backbone-qwen35-0.8b-base

Qwen/Qwen3.5-0.8B-Base.

In [ ]:
TAG = 'backbone-qwen35-0.8b-base'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']                  # rep 1 each
CHECK_SMOKE = ['d3_goemotions', 'd2_banking77']      # smoke with the self-check on
PLAIN_SMOKE = []      # smoke with the self-check off (may be empty)
EXTRA = {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_KEY):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_KEY is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### backbone-qwen35-2b-base

Qwen/Qwen3.5-2B-Base.

In [ ]:
TAG = 'backbone-qwen35-2b-base'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']                  # rep 1 each
CHECK_SMOKE = ['d3_goemotions', 'd2_banking77']      # smoke with the self-check on
PLAIN_SMOKE = []      # smoke with the self-check off (may be empty)
EXTRA = {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_KEY):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_KEY is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### backbone-qwen35-9b-base

Qwen/Qwen3.5-9B-Base.

In [ ]:
TAG = 'backbone-qwen35-9b-base'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']                  # rep 1 each
CHECK_SMOKE = ['d3_goemotions', 'd2_banking77']      # smoke with the self-check on
PLAIN_SMOKE = []      # smoke with the self-check off (may be empty)
EXTRA = {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_KEY):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_KEY is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### backbone-qwen35-9b

Qwen/Qwen3.5-9B.

In [ ]:
TAG = 'backbone-qwen35-9b'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d3_goemotions', 'd2_banking77']                  # rep 1 each
CHECK_SMOKE = ['d3_goemotions', 'd2_banking77']      # smoke with the self-check on
PLAIN_SMOKE = []      # smoke with the self-check off (may be empty)
EXTRA = {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_KEY):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_KEY is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


## Part A (c): comparator-open on d3_goemotions and d2_banking77

Qwen3-14B-AWQ through vLLM 0.30.0 with the settings of its earlier runs (greedy, thinking off, verbalized JSON). No batch-1 latency sample: `adapters.py` registers no batch-1 tag for this model. Switch `RUN_COMPARATOR_OPEN`.

In [ ]:
CELL = 'comparator-open'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
PLAN_TAGS = [('comparator-open', ['d3_goemotions', 'd2_banking77'])]          # [(tag, conditions)] in run order, rep 1
SMOKE_CONDS = ['d3_goemotions', 'd2_banking77']
SMOKE_N = 3
# share of smoke replies that may be error lines before a tag is stopped (0: none may be)
MAX_ERR_SHARE = 0.0
# retry conditions with error lines (True), or only conditions with requests that got no line
# at all (False: a sampled reply that fails to parse fails again under the fixed seed)
RETRY_ERRORS = True
REPO = 'Qwen/Qwen3-14B-AWQ'
NEEDS_A100 = False
FREE_AFTER = False          # this cell is the last one that needs REPO on this part
VLLM_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO',
              # greedy decoding does not need the FlashInfer sampler
              'VLLM_USE_FLASHINFER_SAMPLER': '0'}

def smoke_per_question(sd, tag):
    """Engine time per question from the smoke answers (batch share, no engine start)."""
    per_q = []
    for c in SMOKE_CONDS:
        p = f'{sd}/{tag}/{c}__rep1.jsonl'
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for l in fh:
                j = json.loads(l)
                if j.get('error'):
                    continue
                for a in j['answers'].values():
                    r = a.get('raw') or {}
                    if r.get('batch_wall_s') is not None and r.get('n_requests'):
                        per_q.append(r['batch_wall_s'] / r['n_requests'])
    return sum(per_q) / len(per_q) if per_q else None

if not (RUN_PART_A and RUN_COMPARATOR_OPEN):
    print(f'{CELL}: skipped (switch RUN_PART_A and RUN_COMPARATOR_OPEN is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{PLAN_TAGS[0][0]}/run_log_{PLAN_TAGS[0][0]}.txt'
    print(f'{CELL}: free disk under /content {free_gb():.0f} GB', flush=True)
    rc = stream(['bash', '-c', INSTALL + f'/content/envs/{ENV_NAME}/bin/python -c "import vllm, torch; '
                 f'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], first_log)
    if rc:
        for tag, _conds in PLAN_TAGS:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
    else:
        go = []
        for tag, conds in PLAN_TAGS:
            if not (SMOKE_CONDS and (True)):
                go.append((tag, conds))
                continue
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: smoke test, {SMOKE_N} requests each of {SMOKE_CONDS} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, SMOKE_CONDS, log, answers_dir=sd, limit=SMOKE_N,
                                 extra=VLLM_EXTRA)
                errs, _ = smoke_report(sd)
                n_ok_s, n_err_s, outs = smoke_counts(sd)
            except Exception:
                rc, dt, errs, n_ok_s, n_err_s, outs = -1, 0.0, [traceback.format_exc()], 0, 1, []
            share = n_err_s / max(n_ok_s + n_err_s, 1)
            s = smoke_per_question(sd, tag)
            n_q = sum(EXPECTED[c]['decisions'] for c in conds)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, {n_ok_s} '
                  f'answered, {n_err_s} error lines)', flush=True)
            if errs:
                print(f'{tag}: first smoke error: {errs[0]}', flush=True)
            if outs:
                print(f'{tag}: mean output tokens per question in the smoke test '
                      f'{sum(outs) / len(outs):.0f} (max {max(outs)}) -> about '
                      f'{sum(outs) / len(outs) * n_q / 1e6:.1f} M output tokens for the {n_q} '
                      f'questions', flush=True)
            if s is not None:
                print(f'{tag}: {s:.2f} s per question in the smoke batches -> about '
                      f'{s * n_q / 60:.0f} min for the {n_q} questions; small smoke batches '
                      f'overstate batched time', flush=True)
            run_log.setdefault('models', {}).setdefault(tag, {}).update(
                {'smoke_s_incl_load': dt, 'smoke_s_per_question': s,
                 'smoke_answered': n_ok_s, 'smoke_error_lines': n_err_s,
                 'smoke_mean_output_tokens': sum(outs) / len(outs) if outs else None})
            if rc or n_ok_s == 0 or share > MAX_ERR_SHARE:
                mark(tag, False, f'smoke test failed (exit {rc}, {n_err_s} of '
                                 f'{n_ok_s + n_err_s} replies are error lines): {errs[:1]}')
            else:
                go.append((tag, conds))
        for tag, conds in go:
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: {len(conds)} conditions, '
                  f'{sum(EXPECTED[c]["requests"] for c in conds)} requests: {conds} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, conds, log, extra=VLLM_EXTRA)
                print(f'[{tag}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
                run_log.setdefault('models', {}).setdefault(tag, {})['full_min'] = dt / 60
                save_run_log()
                # a crash in the one-process run leaves later conditions incomplete; each is
                # retried in its own process, finished requests are skipped
                for c in (incomplete(tag, conds) if RETRY_ERRORS else unfinished(tag, conds)):
                    rc, dt = harness(ENV_NAME, tag, [c], log, extra=VLLM_EXTRA)
                    print(f'[{tag}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                missing = incomplete(tag, conds)
                detail = ', '.join(f'{c}: {n_ok(tag, c)}/{EXPECTED[c]["requests"]}' for c in missing)
                mark(tag, not missing, f'incomplete: {detail}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
        # the downloaded weights of this model only, once every tag of it is done
        if (NEEDS_A100 and FREE_AFTER and FREE_DISK_AFTER_DONE
                and all(is_done(t) for t, _c in PLAN_TAGS)):
            cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
            shutil.rmtree(cache, ignore_errors=True)
            print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### comparator-open-b1: batch-1 latency

The first 40 requests of `d1_neutral`, one request at a time, in the same `vllm_env` env, as the second revision did for Qwen3.6-27B. Optional in the status: a failure never blocks `ALL_DONE_revision4`.

In [ ]:
CELL = 'comparator-open-b1'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
B1_TAGS = ['comparator-open-b1']             # each runs the first LIMIT requests of COND one request at a time
COND, LIMIT = 'd1_neutral', 40
REPO = 'Qwen/Qwen3-14B-AWQ'
NEEDS_A100 = False
MAIN_TAGS = ['comparator-open']           # the batched tags of this model on this part (for the disk cleanup)
B1_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO', 'VLLM_USE_FLASHINFER_SAMPLER': '0'}

if not (RUN_PART_A and RUN_COMPARATOR_OPEN):
    print(f'{CELL}: skipped (switch RUN_PART_A and RUN_COMPARATOR_OPEN is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{B1_TAGS[0]}/run_log_{B1_TAGS[0]}.txt'
    rc = stream(['bash', '-c', INSTALL], first_log)     # the env of the batched cell, reused
    for tag in B1_TAGS:
        log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
        path = f'{ANSWERS_DIR}/{tag}/{COND}__rep1.jsonl'
        if rc:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
            continue
        print(f'=== {tag}: batch-1 latency sample, the first {LIMIT} requests of {COND} ===',
              flush=True)
        try:
            rc_h, dt = harness(ENV_NAME, tag, [COND], log, limit=LIMIT, extra=B1_EXTRA)
            ok, err = answer_counts(path)
            lats = []
            if os.path.exists(path):
                with open(path, encoding='utf-8') as fh:
                    for l in fh:
                        j = json.loads(l)
                        if not j.get('error') and j.get('latency_s') is not None:
                            lats.append(j['latency_s'])
            med = sorted(lats)[len(lats) // 2] if lats else None
            print(f'[{tag}] exit {rc_h}, {ok}/{LIMIT} answered, {err} error lines, median latency '
                  f'{med} s per request, {dt / 60:.1f} min including the engine start', flush=True)
            run_log.setdefault('models', {})[tag] = {'answered': ok, 'error_lines': err,
                                                     'median_latency_s': med, 'min': dt / 60}
            mark(tag, ok >= LIMIT, '' if ok >= LIMIT else f'{ok}/{LIMIT} answered')
        except Exception:
            mark(tag, False, traceback.format_exc())
        save_run_log()
    # this is the last cell that needs the weights on this part
    if (NEEDS_A100 and FREE_DISK_AFTER_DONE
            and all(is_done(t) for t in MAIN_TAGS + B1_TAGS)):
        cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
        shutil.rmtree(cache, ignore_errors=True)
        print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


## Part A (d): 9B backbones scored on option descriptions

`P4_DESC_ROWS=4` rows per scoring pass. In the third revision both smoke tests ran out of memory in the self-check on `d2_k150`: the check recomputes each description with a full-sequence pass and a float64 log-softmax over every prompt position (2.4 GiB for a 1300-token prompt), which does not fit next to 9B weights on an L4. The self-check therefore runs on `d1_neutral` and `d3_emotion`, where it passed before, and `d2_k150` and `d3_conv_go_awry` (prompts up to 8,000 tokens) get a 2-request smoke test with the check off, on the cached path the full run uses. Then all six conditions run in one process; a condition left incomplete is retried on its own. Switch `RUN_BACKBONE_DESC`.

### backbone-desc-qwen35-9b-base

Qwen/Qwen3.5-9B-Base, the base of kev-9b.

In [ ]:
TAG = 'backbone-desc-qwen35-9b-base'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']                  # rep 1 each
CHECK_SMOKE = ['d1_neutral', 'd3_emotion']      # smoke with the self-check on
PLAIN_SMOKE = ['d2_k150', 'd3_conv_go_awry']      # smoke with the self-check off (may be empty)
EXTRA = {'P4_DESC_ROWS': '4', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_DESC):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_DESC is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


### backbone-desc-qwen35-9b

Qwen/Qwen3.5-9B (post-trained), the base of nimble-9b.

In [ ]:
TAG = 'backbone-desc-qwen35-9b'
ENV_NAME = 'backbone'
INSTALL = "set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/backbone --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 'accelerate>=1.15' huggingface_hub python-dotenv httpx numpy scipy\n(export UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/backbone/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match flash-linear-attention || echo 'optional package failed: flash-linear-attention')\n"
CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']                  # rep 1 each
CHECK_SMOKE = ['d1_neutral', 'd3_emotion']      # smoke with the self-check on
PLAIN_SMOKE = ['d2_k150', 'd3_conv_go_awry']      # smoke with the self-check off (may be empty)
EXTRA = {'P4_DESC_ROWS': '4', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}

if not (RUN_PART_A and RUN_BACKBONE_DESC):
    print(f'{TAG}: skipped (switch RUN_PART_A and RUN_BACKBONE_DESC is off)')
else:
    LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
    SMOKE_DIR = f'/content/smoke/{TAG}'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    try:
        print(f'=== {TAG}: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # For description scoring the full-sequence self-check takes a float64 log-softmax over
        # every prompt position, 2.4 GiB on a d2_k150 prompt, which does not fit next to 9B
        # weights on an L4. It runs on d1_neutral and d3_emotion there; d2_k150 and
        # d3_conv_go_awry are smoke-tested with the check off.
        print(f'=== {TAG}: smoke test with the self-check, 2 requests each of {CHECK_SMOKE} ===',
              flush=True)
        rc1, dt1 = harness(ENV_NAME, TAG, CHECK_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                           extra=dict(EXTRA, P4_OPTSCORE_CHECK='100'))
        errs, dmax = smoke_report(SMOKE_DIR)
        rc2, dt2 = 0, 0.0
        if PLAIN_SMOKE:
            print(f'=== {TAG}: smoke test without the self-check, 2 requests of {PLAIN_SMOKE} ===',
                  flush=True)
            rc2, dt2 = harness(ENV_NAME, TAG, PLAIN_SMOKE, LOG, answers_dir=SMOKE_DIR, limit=2,
                               extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            errs, _ = smoke_report(SMOKE_DIR)
        print(f'{TAG}: smoke {dt1 + dt2:.0f} s including the model loads; largest self-check '
              f'difference {dmax}', flush=True)
        if rc1 or rc2 or errs:
            raise RuntimeError(f'smoke test failed (exit {rc1}/{rc2}, {len(errs)} error lines, '
                               f'first: {errs[:1]}); see {LOG}')
        if dmax is None:
            raise RuntimeError(f'the smoke answers carry no self-check value; see {LOG}')
        n_req = sum(EXPECTED[c]['requests'] for c in CONDS)
        run_log.setdefault('models', {})[TAG] = {'smoke_s_incl_loads': dt1 + dt2,
                                                 'requests': n_req,
                                                 'smoke_check_max_abs_dprob': dmax}
        print(f'=== {TAG}: full run, {len(CONDS)} conditions, {n_req} requests ===', flush=True)
        rc, dt = harness(ENV_NAME, TAG, CONDS, LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
        print(f'[{TAG}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
        run_log['models'][TAG]['full_min'] = dt / 60
        save_run_log()
        # a condition left incomplete (a crash, a disconnect) is retried on its own
        for c in incomplete(TAG, CONDS):
            rc, dt = harness(ENV_NAME, TAG, [c], LOG, extra=dict(EXTRA, P4_OPTSCORE_CHECK='0'))
            print(f'[{TAG}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
        missing = incomplete(TAG, CONDS)
        mark(TAG, not missing, f'incomplete: {missing}' if missing else '')
    except Exception:
        mark(TAG, False, traceback.format_exc())
    finally:
        save_run_log()
        gc.collect()
        gpu_memory()


## Part A (e): optional Jev latency from the cloud

`harness.py --model jev --cond d1_neutral --rep 6 --limit 40` through `typesafe-sdk` 0.7.1, the SDK version of the local runs, with the model pinned to `jev-1.13.0`. Reps 1 to 5 exist locally, so this is rep 6. The API key is read from the Colab secret `TYPESAFE_API_KEY` and handed to the harness process through its environment only; it is never printed or written to a file. Without the secret the cell says so and skips. Switch `RUN_JEV_LATENCY`.

In [ ]:
TAG = 'jev-1.13.0'
ENV_NAME = 'jev'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/jev --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/jev/bin/python -q  typesafe-sdk==0.7.1 python-dotenv httpx numpy\n'
COND, REP, LIMIT = 'd1_neutral', 6, 40

if not (RUN_PART_A and RUN_JEV_LATENCY):
    print(f'Jev latency: skipped (RUN_PART_A = {RUN_PART_A}, RUN_JEV_LATENCY = {RUN_JEV_LATENCY})')
else:
    _key = None
    try:
        from google.colab import userdata
        _key = userdata.get('TYPESAFE_API_KEY')
    except Exception as _e:
        print('Jev latency: skipped. The Colab secret TYPESAFE_API_KEY is missing, or this '
              f'notebook has no access to it ({type(_e).__name__}). Add it under Secrets (the key '
              'icon on the left) and switch on notebook access to run this cell.')
    if _key is not None and not str(_key).strip():
        print('Jev latency: skipped. The Colab secret TYPESAFE_API_KEY is empty.')
        _key = None
    if _key:
        LOG = f'{ANSWERS_DIR}/{TAG}/run_log_{TAG}.txt'
        path = f'{ANSWERS_DIR}/{TAG}/{COND}__rep{REP}.jsonl'
        try:
            print(f'=== Jev latency: building env {ENV_NAME} ===', flush=True)
            rc = stream(['bash', '-c', INSTALL], LOG)
            if rc:
                raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
            # the key reaches the harness process through its environment only; it is never
            # printed, logged or written to a file
            cmd = [f'/content/envs/{ENV_NAME}/bin/python', f'{SRC_DIR}/harness.py', '--model',
                   'jev', '--cond', COND, '--rep', str(REP), '--limit', str(LIMIT)]
            t0 = time.time()
            rc = stream(cmd, LOG, harness_env(ENV_NAME, ANSWERS_DIR,
                                               {'TYPESAFE_API_KEY': str(_key).strip()}))
            ok, err = answer_counts(path)
            lats = []
            if os.path.exists(path):
                with open(path, encoding='utf-8') as fh:
                    for l in fh:
                        j = json.loads(l)
                        if not j.get('error') and j.get('latency_s') is not None:
                            lats.append(j['latency_s'])
            med = sorted(lats)[len(lats) // 2] if lats else None
            print(f'[{TAG}] {COND} rep {REP}: exit {rc}, {ok}/{LIMIT} answered, {err} error lines, '
                  f'median latency {med} s, {(time.time() - t0) / 60:.1f} min', flush=True)
            run_log.setdefault('models', {})[TAG] = {'answered': ok, 'error_lines': err,
                                                     'median_latency_s': med}
            mark(TAG, ok >= LIMIT, '' if ok >= LIMIT else f'{ok}/{LIMIT} answered')
        except Exception as _e:
            mark(TAG, False, f'{type(_e).__name__}: {_e}')
        finally:
            _key = None
            save_run_log()


## Part B: DeBERTa-v3-large fine-tuned per task

Builds the `b05` env, runs `b05_strong_classifiers.py --smoke` with the real pinned checkpoint on a few rows of every task (one epoch, bf16, the 512-token micro-batches of the D3 tasks) and checks the smoke answers (line count, record fields, revision). Then one process per group in the order of `B05_GROUPS`, each retried once if a condition is incomplete. The smoke run prints the training throughput and an estimate per task. Each condition's answers are written at once, with `<cond>__fit.json` next to them. The `d1` group trains one cross-encoder on the typed-decisions training states against the teacher's soft labels and answers `d1_neutral` and `d1_calib`.

In [ ]:
ENV_NAME = 'b05'
PYBIN = f'/content/envs/{ENV_NAME}/bin/python'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache; uv venv /content/envs/b05 --python 3.12 -q --allow-existing\nexport UV_CACHE_DIR=/content/uv-cache; uv pip install --python /content/envs/b05/bin/python -q --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match torch==2.8.0 transformers==5.17.0 sentencepiece==0.2.2 protobuf huggingface_hub numpy\n'
B05 = f'{SRC_DIR}/b05_strong_classifiers.py'
GROUP_TAG = {'clinc': 'baseline-deberta-large-clinc', 'goemo': 'baseline-deberta-large-goemo', 'banking': 'baseline-deberta-large-banking', 'd3': 'baseline-deberta-large-d3', 'd1': 'baseline-deberta-large-d1'}      # group -> tag
TAG_CONDS = {'baseline-deberta-large-clinc': ['d2_k150'], 'baseline-deberta-large-goemo': ['d3_goemotions'], 'baseline-deberta-large-banking': ['d2_banking77'], 'baseline-deberta-large-d3': ['d3_emotion', 'd3_wiki_politeness', 'd3_conv_go_awry', 'd3_wiki_corpus'], 'baseline-deberta-large-d1': ['d1_neutral', 'd1_calib']}      # tag -> conditions
B05_MODEL, B05_REV = 'microsoft/deberta-v3-large', '64a8c8eab3e352a784c658aef62be1662607476f'
SMOKE_TEST_N = 6     # answered items per condition in the smoke run
FIELDS = ['answers', 'condition', 'key', 'latency_s', 'model', 'model_returned', 'rep', 'revision', 'run_date', 'served_temperature', 'usage_in']            # the record fields of the earlier baseline answers

def b05_env():
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_B05_DATA'] = B05_DATA_DIR
    env['P4_ANSWERS'] = ANSWERS_DIR
    env['PATH'] = f'/content/envs/{ENV_NAME}/bin:' + env.get('PATH', '')
    return env

if not RUN_PART_B:
    print(f'part B skipped (RUN_PART_B = {RUN_PART_B})')
else:
    groups = [g for g in B05_GROUPS if g in GROUP_TAG]
    LOG = f'{ANSWERS_DIR}/run_log_b05.txt'
    SMOKE_DIR = '/content/smoke/b05'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    smoke_ok = False
    try:
        print(f'=== part B: building env {ENV_NAME} ===', flush=True)
        rc = stream(['bash', '-c', INSTALL], LOG)
        if rc:
            raise RuntimeError(f'env build failed (exit {rc}); see {LOG}')
        # the real pinned checkpoint on a few rows of every task, one epoch: tokenizer, bf16
        # training with 512-token micro-batches, answers and the fit file
        print(f'=== part B: smoke test with {B05_MODEL}@{B05_REV[:8]} on groups {groups} ===',
              flush=True)
        t0 = time.time()
        rc = stream([PYBIN, B05, 'run', '--smoke', '--model', B05_MODEL, '--revision', B05_REV,
                     '--groups', ','.join(groups), '--answers', SMOKE_DIR], LOG, env=b05_env())
        problems = []
        for g in groups:
            tag = GROUP_TAG[g]
            for c in TAG_CONDS[tag]:
                p = f'{SMOKE_DIR}/{tag}/{c}__rep1.jsonl'
                if not os.path.exists(p) or not os.path.exists(f'{SMOKE_DIR}/{tag}/{c}__fit.json'):
                    problems.append(f'{tag}/{c}: missing')
                    continue
                with open(p, encoding='utf-8') as fh:
                    recs = [json.loads(l) for l in fh if l.strip()]
                if len(recs) != SMOKE_TEST_N or any(r.get('error') for r in recs):
                    problems.append(f'{tag}/{c}: {len(recs)} lines')
                elif sorted(recs[0]) != FIELDS or recs[0]['revision'] != B05_REV:
                    problems.append(f'{tag}/{c}: fields {sorted(recs[0])}, revision {recs[0]["revision"]}')
        run_log.setdefault('models', {})['b05'] = {'smoke_min': (time.time() - t0) / 60}
        if rc or problems:
            raise RuntimeError(f'smoke test failed (exit {rc}): {problems}; see {LOG}')
        smoke_ok = True
    except Exception:
        tb = traceback.format_exc()
        for g in groups:
            mark(GROUP_TAG[g], False, tb)
    if smoke_ok:
        for g in groups:
            tag = GROUP_TAG[g]
            try:
                for attempt in (1, 2):
                    print(f'=== {tag}: group {g}, attempt {attempt} ===', flush=True)
                    t0 = time.time()
                    rc = stream([PYBIN, B05, 'run', '--groups', g], LOG, env=b05_env())
                    run_log['models'].setdefault(tag, {})[f'attempt{attempt}_min'] = (time.time() - t0) / 60
                    missing = incomplete(tag, TAG_CONDS[tag])
                    print(f'[{tag}] exit {rc}, {(time.time() - t0) / 60:.1f} min, incomplete: {missing}',
                          flush=True)
                    if not missing:
                        break
                mark(tag, not missing, f'incomplete: {missing}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
    save_run_log()
    gpu_memory()


## Parts C1, C2 and C3: comparators from three model families (A100 80GB)

Part C1 is the Gemma cell (switch `RUN_PART_C1`), Part C2 the Mistral cell (`RUN_PART_C2`), Part C3 the Qwen3.6-27B thinking cell and then the Qwen3.6-27B pair (`RUN_PART_C3`, with `RUN_THINK` and `RUN_QWEN36`). The three can run at the same time on three A100 runtimes. Each cell builds (or reuses) the `vllm_env` env (vLLM 0.30.0). Gemma and Mistral are quantized to FP8 weights on load, as `adapters.py` sets in their profiles, and run every condition Qwen3.6-27B covers. For each of them both modes are smoke-tested first (3 requests each of `d1_neutral`, `d2_k150` and `d3_conv_go_awry`); a mode whose smoke test fails is skipped and the other still runs. Each mode then runs all its conditions in one process, and a condition left incomplete is retried on its own. There are no batch-1 latency samples for Gemma or Mistral: `adapters.py` registers no batch-1 tag for them. With `FREE_DISK_AFTER_DONE`, a model's downloaded weights are removed from the runtime disk once all its tags on that part are done.

### Part C1: Gemma-4-31B-it

Tags `comparator-gemma`, `comparator-gemma-ll` on `d1_neutral`, `d1_calib`, `d2_k150`, `d2_k20`, `d3_conv_go_awry`, `d3_wiki_corpus`, `d3_emotion`, `d3_wiki_politeness`, `d3_goemotions`, `d2_banking77`, `d2_k150_oos`, `d2_k5`, `d2_k50`, `e2_d1_k01`, `e2_d1_kny`, `e2_d1_kswap`, `e2_d1_krand`, `e2_d3_conv_go_awry_k01`, `e2_d3_conv_go_awry_kny`, `e2_d3_conv_go_awry_kswap`, `e2_d3_conv_go_awry_krand`, `e2_d3_wiki_corpus_k01`, `e2_d3_wiki_corpus_kny`, `e2_d3_wiki_corpus_kswap`, `e2_d3_wiki_corpus_krand`.

In [ ]:
CELL = 'Gemma-4-31B-it'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
PLAN_TAGS = [('comparator-gemma', ['d1_neutral', 'd1_calib', 'd2_k150', 'd2_k20', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness', 'd3_goemotions', 'd2_banking77', 'd2_k150_oos', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand']), ('comparator-gemma-ll', ['d1_neutral', 'd1_calib', 'd2_k150', 'd2_k20', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness', 'd3_goemotions', 'd2_banking77', 'd2_k150_oos', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand'])]          # [(tag, conditions)] in run order, rep 1
SMOKE_CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry']
SMOKE_N = 3
# share of smoke replies that may be error lines before a tag is stopped (0: none may be)
MAX_ERR_SHARE = 0.0
# retry conditions with error lines (True), or only conditions with requests that got no line
# at all (False: a sampled reply that fails to parse fails again under the fixed seed)
RETRY_ERRORS = True
REPO = 'google/gemma-4-31B-it'
NEEDS_A100 = True
FREE_AFTER = False          # this cell is the last one that needs REPO on this part
VLLM_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO',
              # greedy decoding does not need the FlashInfer sampler
              'VLLM_USE_FLASHINFER_SAMPLER': '0'}

def smoke_per_question(sd, tag):
    """Engine time per question from the smoke answers (batch share, no engine start)."""
    per_q = []
    for c in SMOKE_CONDS:
        p = f'{sd}/{tag}/{c}__rep1.jsonl'
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for l in fh:
                j = json.loads(l)
                if j.get('error'):
                    continue
                for a in j['answers'].values():
                    r = a.get('raw') or {}
                    if r.get('batch_wall_s') is not None and r.get('n_requests'):
                        per_q.append(r['batch_wall_s'] / r['n_requests'])
    return sum(per_q) / len(per_q) if per_q else None

if not (RUN_PART_C1):
    print(f'{CELL}: skipped (switch RUN_PART_C1 is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{PLAN_TAGS[0][0]}/run_log_{PLAN_TAGS[0][0]}.txt'
    print(f'{CELL}: free disk under /content {free_gb():.0f} GB', flush=True)
    rc = stream(['bash', '-c', INSTALL + f'/content/envs/{ENV_NAME}/bin/python -c "import vllm, torch; '
                 f'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], first_log)
    if rc:
        for tag, _conds in PLAN_TAGS:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
    else:
        go = []
        for tag, conds in PLAN_TAGS:
            if not (SMOKE_CONDS and (True)):
                go.append((tag, conds))
                continue
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: smoke test, {SMOKE_N} requests each of {SMOKE_CONDS} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, SMOKE_CONDS, log, answers_dir=sd, limit=SMOKE_N,
                                 extra=VLLM_EXTRA)
                errs, _ = smoke_report(sd)
                n_ok_s, n_err_s, outs = smoke_counts(sd)
            except Exception:
                rc, dt, errs, n_ok_s, n_err_s, outs = -1, 0.0, [traceback.format_exc()], 0, 1, []
            share = n_err_s / max(n_ok_s + n_err_s, 1)
            s = smoke_per_question(sd, tag)
            n_q = sum(EXPECTED[c]['decisions'] for c in conds)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, {n_ok_s} '
                  f'answered, {n_err_s} error lines)', flush=True)
            if errs:
                print(f'{tag}: first smoke error: {errs[0]}', flush=True)
            if outs:
                print(f'{tag}: mean output tokens per question in the smoke test '
                      f'{sum(outs) / len(outs):.0f} (max {max(outs)}) -> about '
                      f'{sum(outs) / len(outs) * n_q / 1e6:.1f} M output tokens for the {n_q} '
                      f'questions', flush=True)
            if s is not None:
                print(f'{tag}: {s:.2f} s per question in the smoke batches -> about '
                      f'{s * n_q / 60:.0f} min for the {n_q} questions; small smoke batches '
                      f'overstate batched time', flush=True)
            run_log.setdefault('models', {}).setdefault(tag, {}).update(
                {'smoke_s_incl_load': dt, 'smoke_s_per_question': s,
                 'smoke_answered': n_ok_s, 'smoke_error_lines': n_err_s,
                 'smoke_mean_output_tokens': sum(outs) / len(outs) if outs else None})
            if rc or n_ok_s == 0 or share > MAX_ERR_SHARE:
                mark(tag, False, f'smoke test failed (exit {rc}, {n_err_s} of '
                                 f'{n_ok_s + n_err_s} replies are error lines): {errs[:1]}')
            else:
                go.append((tag, conds))
        for tag, conds in go:
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: {len(conds)} conditions, '
                  f'{sum(EXPECTED[c]["requests"] for c in conds)} requests: {conds} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, conds, log, extra=VLLM_EXTRA)
                print(f'[{tag}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
                run_log.setdefault('models', {}).setdefault(tag, {})['full_min'] = dt / 60
                save_run_log()
                # a crash in the one-process run leaves later conditions incomplete; each is
                # retried in its own process, finished requests are skipped
                for c in (incomplete(tag, conds) if RETRY_ERRORS else unfinished(tag, conds)):
                    rc, dt = harness(ENV_NAME, tag, [c], log, extra=VLLM_EXTRA)
                    print(f'[{tag}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                missing = incomplete(tag, conds)
                detail = ', '.join(f'{c}: {n_ok(tag, c)}/{EXPECTED[c]["requests"]}' for c in missing)
                mark(tag, not missing, f'incomplete: {detail}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
        # the downloaded weights of this model only, once every tag of it is done
        if (NEEDS_A100 and FREE_AFTER and FREE_DISK_AFTER_DONE
                and all(is_done(t) for t, _c in PLAN_TAGS)):
            cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
            shutil.rmtree(cache, ignore_errors=True)
            print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### Part C1: Gemma-4-31B-it, batch-1 latency

Tags `comparator-gemma-b1` and `comparator-gemma-ll-b1`: the first 40 requests of `d1_neutral`, one request at a time, as the second revision did for Qwen3.6-27B. Optional in the status: a failure never blocks `ALL_DONE_revision4`. This cell removes the model's weights afterwards when `FREE_DISK_AFTER_DONE` is on and every tag of the model is done.

In [ ]:
CELL = 'Gemma-4-31B-it, batch 1'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
B1_TAGS = ['comparator-gemma-b1', 'comparator-gemma-ll-b1']             # each runs the first LIMIT requests of COND one request at a time
COND, LIMIT = 'd1_neutral', 40
REPO = 'google/gemma-4-31B-it'
NEEDS_A100 = True
MAIN_TAGS = ['comparator-gemma', 'comparator-gemma-ll']           # the batched tags of this model on this part (for the disk cleanup)
B1_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO', 'VLLM_USE_FLASHINFER_SAMPLER': '0'}

if not (RUN_PART_C1):
    print(f'{CELL}: skipped (switch RUN_PART_C1 is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{B1_TAGS[0]}/run_log_{B1_TAGS[0]}.txt'
    rc = stream(['bash', '-c', INSTALL], first_log)     # the env of the batched cell, reused
    for tag in B1_TAGS:
        log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
        path = f'{ANSWERS_DIR}/{tag}/{COND}__rep1.jsonl'
        if rc:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
            continue
        print(f'=== {tag}: batch-1 latency sample, the first {LIMIT} requests of {COND} ===',
              flush=True)
        try:
            rc_h, dt = harness(ENV_NAME, tag, [COND], log, limit=LIMIT, extra=B1_EXTRA)
            ok, err = answer_counts(path)
            lats = []
            if os.path.exists(path):
                with open(path, encoding='utf-8') as fh:
                    for l in fh:
                        j = json.loads(l)
                        if not j.get('error') and j.get('latency_s') is not None:
                            lats.append(j['latency_s'])
            med = sorted(lats)[len(lats) // 2] if lats else None
            print(f'[{tag}] exit {rc_h}, {ok}/{LIMIT} answered, {err} error lines, median latency '
                  f'{med} s per request, {dt / 60:.1f} min including the engine start', flush=True)
            run_log.setdefault('models', {})[tag] = {'answered': ok, 'error_lines': err,
                                                     'median_latency_s': med, 'min': dt / 60}
            mark(tag, ok >= LIMIT, '' if ok >= LIMIT else f'{ok}/{LIMIT} answered')
        except Exception:
            mark(tag, False, traceback.format_exc())
        save_run_log()
    # this is the last cell that needs the weights on this part
    if (NEEDS_A100 and FREE_DISK_AFTER_DONE
            and all(is_done(t) for t in MAIN_TAGS + B1_TAGS)):
        cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
        shutil.rmtree(cache, ignore_errors=True)
        print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### Part C2: Mistral-Small-24B-Instruct-2501

Tags `comparator-mistral`, `comparator-mistral-ll` on `d1_neutral`, `d1_calib`, `d2_k150`, `d2_k20`, `d3_conv_go_awry`, `d3_wiki_corpus`, `d3_emotion`, `d3_wiki_politeness`, `d3_goemotions`, `d2_banking77`, `d2_k150_oos`, `d2_k5`, `d2_k50`, `e2_d1_k01`, `e2_d1_kny`, `e2_d1_kswap`, `e2_d1_krand`, `e2_d3_conv_go_awry_k01`, `e2_d3_conv_go_awry_kny`, `e2_d3_conv_go_awry_kswap`, `e2_d3_conv_go_awry_krand`, `e2_d3_wiki_corpus_k01`, `e2_d3_wiki_corpus_kny`, `e2_d3_wiki_corpus_kswap`, `e2_d3_wiki_corpus_krand`.

In [ ]:
CELL = 'Mistral-Small-24B-Instruct-2501'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
PLAN_TAGS = [('comparator-mistral', ['d1_neutral', 'd1_calib', 'd2_k150', 'd2_k20', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness', 'd3_goemotions', 'd2_banking77', 'd2_k150_oos', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand']), ('comparator-mistral-ll', ['d1_neutral', 'd1_calib', 'd2_k150', 'd2_k20', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness', 'd3_goemotions', 'd2_banking77', 'd2_k150_oos', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand'])]          # [(tag, conditions)] in run order, rep 1
SMOKE_CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry']
SMOKE_N = 3
# share of smoke replies that may be error lines before a tag is stopped (0: none may be)
MAX_ERR_SHARE = 0.0
# retry conditions with error lines (True), or only conditions with requests that got no line
# at all (False: a sampled reply that fails to parse fails again under the fixed seed)
RETRY_ERRORS = True
REPO = 'mistralai/Mistral-Small-24B-Instruct-2501'
NEEDS_A100 = True
FREE_AFTER = False          # this cell is the last one that needs REPO on this part
VLLM_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO',
              # greedy decoding does not need the FlashInfer sampler
              'VLLM_USE_FLASHINFER_SAMPLER': '0'}

def smoke_per_question(sd, tag):
    """Engine time per question from the smoke answers (batch share, no engine start)."""
    per_q = []
    for c in SMOKE_CONDS:
        p = f'{sd}/{tag}/{c}__rep1.jsonl'
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for l in fh:
                j = json.loads(l)
                if j.get('error'):
                    continue
                for a in j['answers'].values():
                    r = a.get('raw') or {}
                    if r.get('batch_wall_s') is not None and r.get('n_requests'):
                        per_q.append(r['batch_wall_s'] / r['n_requests'])
    return sum(per_q) / len(per_q) if per_q else None

if not (RUN_PART_C2):
    print(f'{CELL}: skipped (switch RUN_PART_C2 is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{PLAN_TAGS[0][0]}/run_log_{PLAN_TAGS[0][0]}.txt'
    print(f'{CELL}: free disk under /content {free_gb():.0f} GB', flush=True)
    rc = stream(['bash', '-c', INSTALL + f'/content/envs/{ENV_NAME}/bin/python -c "import vllm, torch; '
                 f'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], first_log)
    if rc:
        for tag, _conds in PLAN_TAGS:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
    else:
        go = []
        for tag, conds in PLAN_TAGS:
            if not (SMOKE_CONDS and (True)):
                go.append((tag, conds))
                continue
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: smoke test, {SMOKE_N} requests each of {SMOKE_CONDS} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, SMOKE_CONDS, log, answers_dir=sd, limit=SMOKE_N,
                                 extra=VLLM_EXTRA)
                errs, _ = smoke_report(sd)
                n_ok_s, n_err_s, outs = smoke_counts(sd)
            except Exception:
                rc, dt, errs, n_ok_s, n_err_s, outs = -1, 0.0, [traceback.format_exc()], 0, 1, []
            share = n_err_s / max(n_ok_s + n_err_s, 1)
            s = smoke_per_question(sd, tag)
            n_q = sum(EXPECTED[c]['decisions'] for c in conds)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, {n_ok_s} '
                  f'answered, {n_err_s} error lines)', flush=True)
            if errs:
                print(f'{tag}: first smoke error: {errs[0]}', flush=True)
            if outs:
                print(f'{tag}: mean output tokens per question in the smoke test '
                      f'{sum(outs) / len(outs):.0f} (max {max(outs)}) -> about '
                      f'{sum(outs) / len(outs) * n_q / 1e6:.1f} M output tokens for the {n_q} '
                      f'questions', flush=True)
            if s is not None:
                print(f'{tag}: {s:.2f} s per question in the smoke batches -> about '
                      f'{s * n_q / 60:.0f} min for the {n_q} questions; small smoke batches '
                      f'overstate batched time', flush=True)
            run_log.setdefault('models', {}).setdefault(tag, {}).update(
                {'smoke_s_incl_load': dt, 'smoke_s_per_question': s,
                 'smoke_answered': n_ok_s, 'smoke_error_lines': n_err_s,
                 'smoke_mean_output_tokens': sum(outs) / len(outs) if outs else None})
            if rc or n_ok_s == 0 or share > MAX_ERR_SHARE:
                mark(tag, False, f'smoke test failed (exit {rc}, {n_err_s} of '
                                 f'{n_ok_s + n_err_s} replies are error lines): {errs[:1]}')
            else:
                go.append((tag, conds))
        for tag, conds in go:
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: {len(conds)} conditions, '
                  f'{sum(EXPECTED[c]["requests"] for c in conds)} requests: {conds} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, conds, log, extra=VLLM_EXTRA)
                print(f'[{tag}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
                run_log.setdefault('models', {}).setdefault(tag, {})['full_min'] = dt / 60
                save_run_log()
                # a crash in the one-process run leaves later conditions incomplete; each is
                # retried in its own process, finished requests are skipped
                for c in (incomplete(tag, conds) if RETRY_ERRORS else unfinished(tag, conds)):
                    rc, dt = harness(ENV_NAME, tag, [c], log, extra=VLLM_EXTRA)
                    print(f'[{tag}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                missing = incomplete(tag, conds)
                detail = ', '.join(f'{c}: {n_ok(tag, c)}/{EXPECTED[c]["requests"]}' for c in missing)
                mark(tag, not missing, f'incomplete: {detail}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
        # the downloaded weights of this model only, once every tag of it is done
        if (NEEDS_A100 and FREE_AFTER and FREE_DISK_AFTER_DONE
                and all(is_done(t) for t, _c in PLAN_TAGS)):
            cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
            shutil.rmtree(cache, ignore_errors=True)
            print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### Part C2: Mistral-Small-24B-Instruct-2501, batch-1 latency

Tags `comparator-mistral-b1` and `comparator-mistral-ll-b1`: the first 40 requests of `d1_neutral`, one request at a time, as the second revision did for Qwen3.6-27B. Optional in the status: a failure never blocks `ALL_DONE_revision4`. This cell removes the model's weights afterwards when `FREE_DISK_AFTER_DONE` is on and every tag of the model is done.

In [ ]:
CELL = 'Mistral-Small-24B-Instruct-2501, batch 1'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
B1_TAGS = ['comparator-mistral-b1', 'comparator-mistral-ll-b1']             # each runs the first LIMIT requests of COND one request at a time
COND, LIMIT = 'd1_neutral', 40
REPO = 'mistralai/Mistral-Small-24B-Instruct-2501'
NEEDS_A100 = True
MAIN_TAGS = ['comparator-mistral', 'comparator-mistral-ll']           # the batched tags of this model on this part (for the disk cleanup)
B1_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO', 'VLLM_USE_FLASHINFER_SAMPLER': '0'}

if not (RUN_PART_C2):
    print(f'{CELL}: skipped (switch RUN_PART_C2 is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{B1_TAGS[0]}/run_log_{B1_TAGS[0]}.txt'
    rc = stream(['bash', '-c', INSTALL], first_log)     # the env of the batched cell, reused
    for tag in B1_TAGS:
        log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
        path = f'{ANSWERS_DIR}/{tag}/{COND}__rep1.jsonl'
        if rc:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
            continue
        print(f'=== {tag}: batch-1 latency sample, the first {LIMIT} requests of {COND} ===',
              flush=True)
        try:
            rc_h, dt = harness(ENV_NAME, tag, [COND], log, limit=LIMIT, extra=B1_EXTRA)
            ok, err = answer_counts(path)
            lats = []
            if os.path.exists(path):
                with open(path, encoding='utf-8') as fh:
                    for l in fh:
                        j = json.loads(l)
                        if not j.get('error') and j.get('latency_s') is not None:
                            lats.append(j['latency_s'])
            med = sorted(lats)[len(lats) // 2] if lats else None
            print(f'[{tag}] exit {rc_h}, {ok}/{LIMIT} answered, {err} error lines, median latency '
                  f'{med} s per request, {dt / 60:.1f} min including the engine start', flush=True)
            run_log.setdefault('models', {})[tag] = {'answered': ok, 'error_lines': err,
                                                     'median_latency_s': med, 'min': dt / 60}
            mark(tag, ok >= LIMIT, '' if ok >= LIMIT else f'{ok}/{LIMIT} answered')
        except Exception:
            mark(tag, False, traceback.format_exc())
        save_run_log()
    # this is the last cell that needs the weights on this part
    if (NEEDS_A100 and FREE_DISK_AFTER_DONE
            and all(is_done(t) for t in MAIN_TAGS + B1_TAGS)):
        cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
        shutil.rmtree(cache, ignore_errors=True)
        print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### Part C3: Qwen3.6-27B-FP8, thinking on

Tags `comparator-open2-think` on `d1_neutral`, `d1_calib`, `d2_k150`, `d3_conv_go_awry`, `d3_wiki_corpus`, `d3_emotion`, `d3_wiki_politeness`, `d2_k150_oos`, `d3_goemotions`, `d2_banking77`.

Thinking on, verbal readout only (`Comparator2ThinkBackend`: up to 8192 output tokens, a 24576-token context, the sampling of `THINK_SAMPLING` with a fixed seed, JSON parsed after the closing think tag). Smoke test: 5 requests each of `d1_neutral`, `d2_k150`; the tag stops if more than half the smoke replies are error lines, and the cell prints the mean output tokens per question and a time estimate. A reply that fails to parse is an error line; under the fixed seed it fails again, so only requests without any line are retried, and such a condition stays INCOMPLETE in the status with its count.

In [ ]:
CELL = 'Qwen3.6-27B-FP8, thinking on'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
PLAN_TAGS = [('comparator-open2-think', ['d1_neutral', 'd1_calib', 'd2_k150', 'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness', 'd2_k150_oos', 'd3_goemotions', 'd2_banking77'])]          # [(tag, conditions)] in run order, rep 1
SMOKE_CONDS = ['d1_neutral', 'd2_k150']
SMOKE_N = 5
# share of smoke replies that may be error lines before a tag is stopped (0: none may be)
MAX_ERR_SHARE = 0.5
# retry conditions with error lines (True), or only conditions with requests that got no line
# at all (False: a sampled reply that fails to parse fails again under the fixed seed)
RETRY_ERRORS = False
REPO = 'Qwen/Qwen3.6-27B-FP8'
NEEDS_A100 = True
FREE_AFTER = False          # this cell is the last one that needs REPO on this part
VLLM_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO',
              # greedy decoding does not need the FlashInfer sampler
              'VLLM_USE_FLASHINFER_SAMPLER': '0'}

def smoke_per_question(sd, tag):
    """Engine time per question from the smoke answers (batch share, no engine start)."""
    per_q = []
    for c in SMOKE_CONDS:
        p = f'{sd}/{tag}/{c}__rep1.jsonl'
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for l in fh:
                j = json.loads(l)
                if j.get('error'):
                    continue
                for a in j['answers'].values():
                    r = a.get('raw') or {}
                    if r.get('batch_wall_s') is not None and r.get('n_requests'):
                        per_q.append(r['batch_wall_s'] / r['n_requests'])
    return sum(per_q) / len(per_q) if per_q else None

if not (RUN_PART_C3 and RUN_THINK):
    print(f'{CELL}: skipped (switch RUN_PART_C3 and RUN_THINK is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{PLAN_TAGS[0][0]}/run_log_{PLAN_TAGS[0][0]}.txt'
    print(f'{CELL}: free disk under /content {free_gb():.0f} GB', flush=True)
    rc = stream(['bash', '-c', INSTALL + f'/content/envs/{ENV_NAME}/bin/python -c "import vllm, torch; '
                 f'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], first_log)
    if rc:
        for tag, _conds in PLAN_TAGS:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
    else:
        go = []
        for tag, conds in PLAN_TAGS:
            if not (SMOKE_CONDS and (True)):
                go.append((tag, conds))
                continue
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: smoke test, {SMOKE_N} requests each of {SMOKE_CONDS} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, SMOKE_CONDS, log, answers_dir=sd, limit=SMOKE_N,
                                 extra=VLLM_EXTRA)
                errs, _ = smoke_report(sd)
                n_ok_s, n_err_s, outs = smoke_counts(sd)
            except Exception:
                rc, dt, errs, n_ok_s, n_err_s, outs = -1, 0.0, [traceback.format_exc()], 0, 1, []
            share = n_err_s / max(n_ok_s + n_err_s, 1)
            s = smoke_per_question(sd, tag)
            n_q = sum(EXPECTED[c]['decisions'] for c in conds)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, {n_ok_s} '
                  f'answered, {n_err_s} error lines)', flush=True)
            if errs:
                print(f'{tag}: first smoke error: {errs[0]}', flush=True)
            if outs:
                print(f'{tag}: mean output tokens per question in the smoke test '
                      f'{sum(outs) / len(outs):.0f} (max {max(outs)}) -> about '
                      f'{sum(outs) / len(outs) * n_q / 1e6:.1f} M output tokens for the {n_q} '
                      f'questions', flush=True)
            if s is not None:
                print(f'{tag}: {s:.2f} s per question in the smoke batches -> about '
                      f'{s * n_q / 60:.0f} min for the {n_q} questions; small smoke batches '
                      f'overstate batched time', flush=True)
            run_log.setdefault('models', {}).setdefault(tag, {}).update(
                {'smoke_s_incl_load': dt, 'smoke_s_per_question': s,
                 'smoke_answered': n_ok_s, 'smoke_error_lines': n_err_s,
                 'smoke_mean_output_tokens': sum(outs) / len(outs) if outs else None})
            if rc or n_ok_s == 0 or share > MAX_ERR_SHARE:
                mark(tag, False, f'smoke test failed (exit {rc}, {n_err_s} of '
                                 f'{n_ok_s + n_err_s} replies are error lines): {errs[:1]}')
            else:
                go.append((tag, conds))
        for tag, conds in go:
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: {len(conds)} conditions, '
                  f'{sum(EXPECTED[c]["requests"] for c in conds)} requests: {conds} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, conds, log, extra=VLLM_EXTRA)
                print(f'[{tag}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
                run_log.setdefault('models', {}).setdefault(tag, {})['full_min'] = dt / 60
                save_run_log()
                # a crash in the one-process run leaves later conditions incomplete; each is
                # retried in its own process, finished requests are skipped
                for c in (incomplete(tag, conds) if RETRY_ERRORS else unfinished(tag, conds)):
                    rc, dt = harness(ENV_NAME, tag, [c], log, extra=VLLM_EXTRA)
                    print(f'[{tag}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                missing = incomplete(tag, conds)
                detail = ', '.join(f'{c}: {n_ok(tag, c)}/{EXPECTED[c]["requests"]}' for c in missing)
                mark(tag, not missing, f'incomplete: {detail}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
        # the downloaded weights of this model only, once every tag of it is done
        if (NEEDS_A100 and FREE_AFTER and FREE_DISK_AFTER_DONE
                and all(is_done(t) for t, _c in PLAN_TAGS)):
            cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
            shutil.rmtree(cache, ignore_errors=True)
            print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


### Part C3: Qwen3.6-27B-FP8

Tags `comparator-open2`, `comparator-open2-ll` on `d3_goemotions`, `d2_banking77`, `d2_k5`, `d2_k50`, `e2_d1_k01`, `e2_d1_kny`, `e2_d1_kswap`, `e2_d1_krand`, `e2_d3_conv_go_awry_k01`, `e2_d3_conv_go_awry_kny`, `e2_d3_conv_go_awry_kswap`, `e2_d3_conv_go_awry_krand`, `e2_d3_wiki_corpus_k01`, `e2_d3_wiki_corpus_kny`, `e2_d3_wiki_corpus_kswap`, `e2_d3_wiki_corpus_krand`.

In [ ]:
CELL = 'Qwen3.6-27B-FP8'
ENV_NAME = 'vllm_env'
INSTALL = 'set -e\nexport UV_CACHE_DIR=/content/uv-cache\nuv venv /content/envs/vllm_env --python 3.12 -q --allow-existing\nuv pip install --python /content/envs/vllm_env/bin/python -q vllm==0.30.0 huggingface_hub python-dotenv httpx numpy ninja\n'
PLAN_TAGS = [('comparator-open2', ['d3_goemotions', 'd2_banking77', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand']), ('comparator-open2-ll', ['d3_goemotions', 'd2_banking77', 'd2_k5', 'd2_k50', 'e2_d1_k01', 'e2_d1_kny', 'e2_d1_kswap', 'e2_d1_krand', 'e2_d3_conv_go_awry_k01', 'e2_d3_conv_go_awry_kny', 'e2_d3_conv_go_awry_kswap', 'e2_d3_conv_go_awry_krand', 'e2_d3_wiki_corpus_k01', 'e2_d3_wiki_corpus_kny', 'e2_d3_wiki_corpus_kswap', 'e2_d3_wiki_corpus_krand'])]          # [(tag, conditions)] in run order, rep 1
SMOKE_CONDS = ['d1_neutral', 'd2_k150', 'd3_conv_go_awry']
SMOKE_N = 3
# share of smoke replies that may be error lines before a tag is stopped (0: none may be)
MAX_ERR_SHARE = 0.0
# retry conditions with error lines (True), or only conditions with requests that got no line
# at all (False: a sampled reply that fails to parse fails again under the fixed seed)
RETRY_ERRORS = True
REPO = 'Qwen/Qwen3.6-27B-FP8'
NEEDS_A100 = True
FREE_AFTER = True          # this cell is the last one that needs REPO on this part
VLLM_EXTRA = {'VLLM_LOGGING_LEVEL': 'INFO',
              # greedy decoding does not need the FlashInfer sampler
              'VLLM_USE_FLASHINFER_SAMPLER': '0'}

def smoke_per_question(sd, tag):
    """Engine time per question from the smoke answers (batch share, no engine start)."""
    per_q = []
    for c in SMOKE_CONDS:
        p = f'{sd}/{tag}/{c}__rep1.jsonl'
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for l in fh:
                j = json.loads(l)
                if j.get('error'):
                    continue
                for a in j['answers'].values():
                    r = a.get('raw') or {}
                    if r.get('batch_wall_s') is not None and r.get('n_requests'):
                        per_q.append(r['batch_wall_s'] / r['n_requests'])
    return sum(per_q) / len(per_q) if per_q else None

if not (RUN_PART_C3 and RUN_QWEN36):
    print(f'{CELL}: skipped (switch RUN_PART_C3 and RUN_QWEN36 is off)')
elif NEEDS_A100 and not PART_C_GPU_OK:
    print(f'{CELL}: skipped, this part needs an A100 or H100 (this runtime: {gpu_info})')
else:
    first_log = f'{ANSWERS_DIR}/{PLAN_TAGS[0][0]}/run_log_{PLAN_TAGS[0][0]}.txt'
    print(f'{CELL}: free disk under /content {free_gb():.0f} GB', flush=True)
    rc = stream(['bash', '-c', INSTALL + f'/content/envs/{ENV_NAME}/bin/python -c "import vllm, torch; '
                 f'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'], first_log)
    if rc:
        for tag, _conds in PLAN_TAGS:
            mark(tag, False, f'vLLM env build failed (exit {rc}); see {first_log}')
    else:
        go = []
        for tag, conds in PLAN_TAGS:
            if not (SMOKE_CONDS and (QWEN36_SMOKE)):
                go.append((tag, conds))
                continue
            sd = f'/content/smoke/{tag}'
            shutil.rmtree(sd, ignore_errors=True)
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: smoke test, {SMOKE_N} requests each of {SMOKE_CONDS} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, SMOKE_CONDS, log, answers_dir=sd, limit=SMOKE_N,
                                 extra=VLLM_EXTRA)
                errs, _ = smoke_report(sd)
                n_ok_s, n_err_s, outs = smoke_counts(sd)
            except Exception:
                rc, dt, errs, n_ok_s, n_err_s, outs = -1, 0.0, [traceback.format_exc()], 0, 1, []
            share = n_err_s / max(n_ok_s + n_err_s, 1)
            s = smoke_per_question(sd, tag)
            n_q = sum(EXPECTED[c]['decisions'] for c in conds)
            print(f'{tag}: smoke {dt:.0f} s including the engine start (exit {rc}, {n_ok_s} '
                  f'answered, {n_err_s} error lines)', flush=True)
            if errs:
                print(f'{tag}: first smoke error: {errs[0]}', flush=True)
            if outs:
                print(f'{tag}: mean output tokens per question in the smoke test '
                      f'{sum(outs) / len(outs):.0f} (max {max(outs)}) -> about '
                      f'{sum(outs) / len(outs) * n_q / 1e6:.1f} M output tokens for the {n_q} '
                      f'questions', flush=True)
            if s is not None:
                print(f'{tag}: {s:.2f} s per question in the smoke batches -> about '
                      f'{s * n_q / 60:.0f} min for the {n_q} questions; small smoke batches '
                      f'overstate batched time', flush=True)
            run_log.setdefault('models', {}).setdefault(tag, {}).update(
                {'smoke_s_incl_load': dt, 'smoke_s_per_question': s,
                 'smoke_answered': n_ok_s, 'smoke_error_lines': n_err_s,
                 'smoke_mean_output_tokens': sum(outs) / len(outs) if outs else None})
            if rc or n_ok_s == 0 or share > MAX_ERR_SHARE:
                mark(tag, False, f'smoke test failed (exit {rc}, {n_err_s} of '
                                 f'{n_ok_s + n_err_s} replies are error lines): {errs[:1]}')
            else:
                go.append((tag, conds))
        for tag, conds in go:
            log = f'{ANSWERS_DIR}/{tag}/run_log_{tag}.txt'
            print(f'=== {tag}: {len(conds)} conditions, '
                  f'{sum(EXPECTED[c]["requests"] for c in conds)} requests: {conds} ===', flush=True)
            try:
                rc, dt = harness(ENV_NAME, tag, conds, log, extra=VLLM_EXTRA)
                print(f'[{tag}] one-process run: exit {rc}, {dt / 60:.1f} min', flush=True)
                run_log.setdefault('models', {}).setdefault(tag, {})['full_min'] = dt / 60
                save_run_log()
                # a crash in the one-process run leaves later conditions incomplete; each is
                # retried in its own process, finished requests are skipped
                for c in (incomplete(tag, conds) if RETRY_ERRORS else unfinished(tag, conds)):
                    rc, dt = harness(ENV_NAME, tag, [c], log, extra=VLLM_EXTRA)
                    print(f'[{tag}] retry {c}: exit {rc}, {dt / 60:.1f} min', flush=True)
                missing = incomplete(tag, conds)
                detail = ', '.join(f'{c}: {n_ok(tag, c)}/{EXPECTED[c]["requests"]}' for c in missing)
                mark(tag, not missing, f'incomplete: {detail}' if missing else '')
            except Exception:
                mark(tag, False, traceback.format_exc())
            save_run_log()
        # the downloaded weights of this model only, once every tag of it is done
        if (NEEDS_A100 and FREE_AFTER and FREE_DISK_AFTER_DONE
                and all(is_done(t) for t, _c in PLAN_TAGS)):
            cache = '/content/hf/hub/models--' + REPO.replace('/', '--')
            shutil.rmtree(cache, ignore_errors=True)
            print(f'{CELL}: removed the downloaded weights in {cache}; free disk {free_gb():.0f} GB')
    save_run_log()
    gpu_memory()


## Status and hand-back

Reads the Drive folder, so it covers every part whichever runtime wrote it. Lists every planned answer file of all four parts with its count of answered requests, error lines, the longest prompt in tokens (backbones) and the recorded revision against the pin, then a summary per part. A part not run yet shows as MISSING. Then merges the run logs of all runtimes into `run_log_revision4.json` and writes `answers_revision4.zip` into `My Drive/Jev/paper4/colab/`. Run this cell last on the runtime that finishes last; files another runtime wrote in the last few minutes may take a moment to appear, so rerun it if a finished part shows as MISSING.

In [ ]:
import datetime, glob, uuid, zipfile

PLANNED = [['A', 'laya-en', 'd3_goemotions', 1, 492, False], ['A', 'laya-en', 'd2_banking77', 1, 616, False], ['A', 'laya-ml', 'd3_goemotions', 1, 492, False], ['A', 'laya-ml', 'd2_banking77', 1, 616, False], ['A', 'kev-0.8b', 'd3_goemotions', 1, 492, False], ['A', 'kev-0.8b', 'd2_banking77', 1, 616, False], ['A', 'decider-2b', 'd3_goemotions', 1, 492, False], ['A', 'decider-2b', 'd2_banking77', 1, 616, False], ['A', 'this-that-1.0', 'd3_goemotions', 1, 492, False], ['A', 'this-that-1.0', 'd2_banking77', 1, 616, False], ['A', 'kev-9b', 'd3_goemotions', 1, 492, False], ['A', 'kev-9b', 'd2_banking77', 1, 616, False], ['A', 'nimble-9b', 'd3_goemotions', 1, 492, False], ['A', 'nimble-9b', 'd2_banking77', 1, 616, False], ['A', 'backbone-qwen35-0.8b-base', 'd3_goemotions', 1, 492, False], ['A', 'backbone-qwen35-0.8b-base', 'd2_banking77', 1, 616, False], ['A', 'backbone-qwen35-2b-base', 'd3_goemotions', 1, 492, False], ['A', 'backbone-qwen35-2b-base', 'd2_banking77', 1, 616, False], ['A', 'backbone-qwen35-9b-base', 'd3_goemotions', 1, 492, False], ['A', 'backbone-qwen35-9b-base', 'd2_banking77', 1, 616, False], ['A', 'backbone-qwen35-9b', 'd3_goemotions', 1, 492, False], ['A', 'backbone-qwen35-9b', 'd2_banking77', 1, 616, False], ['A', 'comparator-open', 'd3_goemotions', 1, 492, False], ['A', 'comparator-open', 'd2_banking77', 1, 616, False], ['A', 'comparator-open-b1', 'd1_neutral', 1, 40, True], ['A', 'backbone-desc-qwen35-9b-base', 'd1_neutral', 1, 400, False], ['A', 'backbone-desc-qwen35-9b-base', 'd2_k150', 1, 800, False], ['A', 'backbone-desc-qwen35-9b-base', 'd3_conv_go_awry', 1, 500, False], ['A', 'backbone-desc-qwen35-9b-base', 'd3_wiki_corpus', 1, 500, False], ['A', 'backbone-desc-qwen35-9b-base', 'd3_emotion', 1, 498, False], ['A', 'backbone-desc-qwen35-9b-base', 'd3_wiki_politeness', 1, 498, False], ['A', 'backbone-desc-qwen35-9b', 'd1_neutral', 1, 400, False], ['A', 'backbone-desc-qwen35-9b', 'd2_k150', 1, 800, False], ['A', 'backbone-desc-qwen35-9b', 'd3_conv_go_awry', 1, 500, False], ['A', 'backbone-desc-qwen35-9b', 'd3_wiki_corpus', 1, 500, False], ['A', 'backbone-desc-qwen35-9b', 'd3_emotion', 1, 498, False], ['A', 'backbone-desc-qwen35-9b', 'd3_wiki_politeness', 1, 498, False], ['A', 'jev-1.13.0', 'd1_neutral', 6, 40, True], ['B', 'baseline-deberta-large-clinc', 'd2_k150', 1, 800, False], ['B', 'baseline-deberta-large-goemo', 'd3_goemotions', 1, 492, False], ['B', 'baseline-deberta-large-banking', 'd2_banking77', 1, 616, False], ['B', 'baseline-deberta-large-d3', 'd3_emotion', 1, 498, False], ['B', 'baseline-deberta-large-d3', 'd3_wiki_politeness', 1, 498, False], ['B', 'baseline-deberta-large-d3', 'd3_conv_go_awry', 1, 500, False], ['B', 'baseline-deberta-large-d3', 'd3_wiki_corpus', 1, 500, False], ['B', 'baseline-deberta-large-d1', 'd1_neutral', 1, 400, False], ['B', 'baseline-deberta-large-d1', 'd1_calib', 1, 300, False], ['C1', 'comparator-gemma', 'd1_neutral', 1, 400, False], ['C1', 'comparator-gemma', 'd1_calib', 1, 300, False], ['C1', 'comparator-gemma', 'd2_k150', 1, 800, False], ['C1', 'comparator-gemma', 'd2_k20', 1, 600, False], ['C1', 'comparator-gemma', 'd3_conv_go_awry', 1, 500, False], ['C1', 'comparator-gemma', 'd3_wiki_corpus', 1, 500, False], ['C1', 'comparator-gemma', 'd3_emotion', 1, 498, False], ['C1', 'comparator-gemma', 'd3_wiki_politeness', 1, 498, False], ['C1', 'comparator-gemma', 'd3_goemotions', 1, 492, False], ['C1', 'comparator-gemma', 'd2_banking77', 1, 616, False], ['C1', 'comparator-gemma', 'd2_k150_oos', 1, 800, False], ['C1', 'comparator-gemma', 'd2_k5', 1, 600, False], ['C1', 'comparator-gemma', 'd2_k50', 1, 600, False], ['C1', 'comparator-gemma', 'e2_d1_k01', 1, 400, False], ['C1', 'comparator-gemma', 'e2_d1_kny', 1, 400, False], ['C1', 'comparator-gemma', 'e2_d1_kswap', 1, 400, False], ['C1', 'comparator-gemma', 'e2_d1_krand', 1, 400, False], ['C1', 'comparator-gemma', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C1', 'comparator-gemma', 'e2_d3_wiki_corpus_krand', 1, 500, False], ['C1', 'comparator-gemma-ll', 'd1_neutral', 1, 400, False], ['C1', 'comparator-gemma-ll', 'd1_calib', 1, 300, False], ['C1', 'comparator-gemma-ll', 'd2_k150', 1, 800, False], ['C1', 'comparator-gemma-ll', 'd2_k20', 1, 600, False], ['C1', 'comparator-gemma-ll', 'd3_conv_go_awry', 1, 500, False], ['C1', 'comparator-gemma-ll', 'd3_wiki_corpus', 1, 500, False], ['C1', 'comparator-gemma-ll', 'd3_emotion', 1, 498, False], ['C1', 'comparator-gemma-ll', 'd3_wiki_politeness', 1, 498, False], ['C1', 'comparator-gemma-ll', 'd3_goemotions', 1, 492, False], ['C1', 'comparator-gemma-ll', 'd2_banking77', 1, 616, False], ['C1', 'comparator-gemma-ll', 'd2_k150_oos', 1, 800, False], ['C1', 'comparator-gemma-ll', 'd2_k5', 1, 600, False], ['C1', 'comparator-gemma-ll', 'd2_k50', 1, 600, False], ['C1', 'comparator-gemma-ll', 'e2_d1_k01', 1, 400, False], ['C1', 'comparator-gemma-ll', 'e2_d1_kny', 1, 400, False], ['C1', 'comparator-gemma-ll', 'e2_d1_kswap', 1, 400, False], ['C1', 'comparator-gemma-ll', 'e2_d1_krand', 1, 400, False], ['C1', 'comparator-gemma-ll', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C1', 'comparator-gemma-ll', 'e2_d3_wiki_corpus_krand', 1, 500, False], ['C1', 'comparator-gemma-b1', 'd1_neutral', 1, 40, True], ['C1', 'comparator-gemma-ll-b1', 'd1_neutral', 1, 40, True], ['C2', 'comparator-mistral', 'd1_neutral', 1, 400, False], ['C2', 'comparator-mistral', 'd1_calib', 1, 300, False], ['C2', 'comparator-mistral', 'd2_k150', 1, 800, False], ['C2', 'comparator-mistral', 'd2_k20', 1, 600, False], ['C2', 'comparator-mistral', 'd3_conv_go_awry', 1, 500, False], ['C2', 'comparator-mistral', 'd3_wiki_corpus', 1, 500, False], ['C2', 'comparator-mistral', 'd3_emotion', 1, 498, False], ['C2', 'comparator-mistral', 'd3_wiki_politeness', 1, 498, False], ['C2', 'comparator-mistral', 'd3_goemotions', 1, 492, False], ['C2', 'comparator-mistral', 'd2_banking77', 1, 616, False], ['C2', 'comparator-mistral', 'd2_k150_oos', 1, 800, False], ['C2', 'comparator-mistral', 'd2_k5', 1, 600, False], ['C2', 'comparator-mistral', 'd2_k50', 1, 600, False], ['C2', 'comparator-mistral', 'e2_d1_k01', 1, 400, False], ['C2', 'comparator-mistral', 'e2_d1_kny', 1, 400, False], ['C2', 'comparator-mistral', 'e2_d1_kswap', 1, 400, False], ['C2', 'comparator-mistral', 'e2_d1_krand', 1, 400, False], ['C2', 'comparator-mistral', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C2', 'comparator-mistral', 'e2_d3_wiki_corpus_krand', 1, 500, False], ['C2', 'comparator-mistral-ll', 'd1_neutral', 1, 400, False], ['C2', 'comparator-mistral-ll', 'd1_calib', 1, 300, False], ['C2', 'comparator-mistral-ll', 'd2_k150', 1, 800, False], ['C2', 'comparator-mistral-ll', 'd2_k20', 1, 600, False], ['C2', 'comparator-mistral-ll', 'd3_conv_go_awry', 1, 500, False], ['C2', 'comparator-mistral-ll', 'd3_wiki_corpus', 1, 500, False], ['C2', 'comparator-mistral-ll', 'd3_emotion', 1, 498, False], ['C2', 'comparator-mistral-ll', 'd3_wiki_politeness', 1, 498, False], ['C2', 'comparator-mistral-ll', 'd3_goemotions', 1, 492, False], ['C2', 'comparator-mistral-ll', 'd2_banking77', 1, 616, False], ['C2', 'comparator-mistral-ll', 'd2_k150_oos', 1, 800, False], ['C2', 'comparator-mistral-ll', 'd2_k5', 1, 600, False], ['C2', 'comparator-mistral-ll', 'd2_k50', 1, 600, False], ['C2', 'comparator-mistral-ll', 'e2_d1_k01', 1, 400, False], ['C2', 'comparator-mistral-ll', 'e2_d1_kny', 1, 400, False], ['C2', 'comparator-mistral-ll', 'e2_d1_kswap', 1, 400, False], ['C2', 'comparator-mistral-ll', 'e2_d1_krand', 1, 400, False], ['C2', 'comparator-mistral-ll', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C2', 'comparator-mistral-ll', 'e2_d3_wiki_corpus_krand', 1, 500, False], ['C2', 'comparator-mistral-b1', 'd1_neutral', 1, 40, True], ['C2', 'comparator-mistral-ll-b1', 'd1_neutral', 1, 40, True], ['C3', 'comparator-open2-think', 'd1_neutral', 1, 400, False], ['C3', 'comparator-open2-think', 'd1_calib', 1, 300, False], ['C3', 'comparator-open2-think', 'd2_k150', 1, 800, False], ['C3', 'comparator-open2-think', 'd3_conv_go_awry', 1, 500, False], ['C3', 'comparator-open2-think', 'd3_wiki_corpus', 1, 500, False], ['C3', 'comparator-open2-think', 'd3_emotion', 1, 498, False], ['C3', 'comparator-open2-think', 'd3_wiki_politeness', 1, 498, False], ['C3', 'comparator-open2-think', 'd2_k150_oos', 1, 800, False], ['C3', 'comparator-open2-think', 'd3_goemotions', 1, 492, False], ['C3', 'comparator-open2-think', 'd2_banking77', 1, 616, False], ['C3', 'comparator-open2', 'd3_goemotions', 1, 492, False], ['C3', 'comparator-open2', 'd2_banking77', 1, 616, False], ['C3', 'comparator-open2', 'd2_k5', 1, 600, False], ['C3', 'comparator-open2', 'd2_k50', 1, 600, False], ['C3', 'comparator-open2', 'e2_d1_k01', 1, 400, False], ['C3', 'comparator-open2', 'e2_d1_kny', 1, 400, False], ['C3', 'comparator-open2', 'e2_d1_kswap', 1, 400, False], ['C3', 'comparator-open2', 'e2_d1_krand', 1, 400, False], ['C3', 'comparator-open2', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C3', 'comparator-open2', 'e2_d3_wiki_corpus_krand', 1, 500, False], ['C3', 'comparator-open2-ll', 'd3_goemotions', 1, 492, False], ['C3', 'comparator-open2-ll', 'd2_banking77', 1, 616, False], ['C3', 'comparator-open2-ll', 'd2_k5', 1, 600, False], ['C3', 'comparator-open2-ll', 'd2_k50', 1, 600, False], ['C3', 'comparator-open2-ll', 'e2_d1_k01', 1, 400, False], ['C3', 'comparator-open2-ll', 'e2_d1_kny', 1, 400, False], ['C3', 'comparator-open2-ll', 'e2_d1_kswap', 1, 400, False], ['C3', 'comparator-open2-ll', 'e2_d1_krand', 1, 400, False], ['C3', 'comparator-open2-ll', 'e2_d3_conv_go_awry_k01', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_conv_go_awry_kny', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_conv_go_awry_kswap', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_conv_go_awry_krand', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_wiki_corpus_k01', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_wiki_corpus_kny', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_wiki_corpus_kswap', 1, 500, False], ['C3', 'comparator-open2-ll', 'e2_d3_wiki_corpus_krand', 1, 500, False]]   # [(part, tag, condition, rep, requests wanted, optional)]

def scan(path):
    ok = err = 0
    revs, ntok, trunc, errs = set(), 0, 0, {}
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            try:
                j = json.loads(line)
            except json.JSONDecodeError:
                continue
            if j.get('error'):
                err += 1
                e = j['error']
                if 'max_len' in e or 'max_model_len' in e or 'budget' in e:
                    trunc += 1
                errs[e[:80]] = errs.get(e[:80], 0) + 1
                continue
            ok += 1
            revs.add(j.get('revision'))
            for a in j['answers'].values():
                ntok = max(ntok, (a.get('raw') or {}).get('n_prompt_tokens') or 0)
    return ok, err, revs, ntok, trunc, errs

status, problems = {}, []
for part, tag, cond, rep, want, optional in PLANNED:
    path = f'{ANSWERS_DIR}/{tag}/{cond}__rep{rep}.jsonl'
    key = f'{tag}/{cond}__rep{rep}'
    if not os.path.exists(path):
        status[key] = f'MISSING (part {part}{", optional" if optional else ""})'
        if not optional:
            problems.append(key)
        continue
    ok, err, revs, ntok, trunc, errs = scan(path)
    rev_ok = revs == {REVISIONS.get(TAG_REPO[tag], TAG_REPO[tag])}
    note = (f'{ok}/{want} answered, {err} error lines ({trunc} over the length limit), '
            f'revision {"ok" if rev_ok else revs}')
    if ntok:
        note += f', longest prompt {ntok} tokens'
    complete = ok >= want and rev_ok
    status[key] = ('OK ' if complete else 'INCOMPLETE ') + note
    if errs:
        status[key] += f', errors: {errs}'
    if not complete and not optional:
        problems.append(key)

for k, v in status.items():
    print(f'{k:62s} {v}')
# every part's files are read from Drive, whichever runtime wrote them; files another runtime
# wrote in the last minutes may not be visible yet, so rerun this cell if one shows as MISSING
print()
for part in sorted({x[0] for x in PLANNED}):
    keys = [f'{x[1]}/{x[2]}__rep{x[3]}' for x in PLANNED if x[0] == part]
    n_ok_part = sum(status[k].startswith('OK') for k in keys)
    n_missing = sum(status[k].startswith('MISSING') for k in keys)
    print(f'part {part:3s}: {n_ok_part}/{len(keys)} files complete, {n_missing} missing')

# the logs of the runtimes, merged read-only into one file; each runtime writes only its own.
# Temporary names carry a random suffix: two runtimes can have the same process id.
merged = {'parts': {}, 'end': datetime.datetime.utcnow().isoformat() + 'Z', 'status': status,
          'revisions_pinned': REVISIONS}
for p in sorted(glob.glob(f'{ANSWERS_DIR}/run_log_revision4_part*.json')):
    try:
        merged['parts'][os.path.basename(p)] = json.load(open(p))
    except Exception as e:
        merged['parts'][os.path.basename(p)] = f'unreadable: {e}'
tmp = f'{ANSWERS_DIR}/run_log_revision4.json.{uuid.uuid4().hex}.tmp'
with open(tmp, 'w') as fh:
    json.dump(merged, fh, indent=2)
os.replace(tmp, f'{ANSWERS_DIR}/run_log_revision4.json')

if problems:
    print(f'\n{len(problems)} file(s) missing or incomplete. A part not run yet shows as MISSING; '
          'otherwise re-run the cell of that model: finished requests are skipped.')
else:
    with open(f'{ANSWERS_DIR}/ALL_DONE_revision4', 'w') as fh:
        fh.write(merged['end'] + '\n')
    print('\nEvery planned answer file is complete. ALL_DONE_revision4 written.')

# the hand-back archive next to the folder on Drive: answer files, fit files of the trained
# baselines, DONE markers and the run logs; console logs and tracebacks stay on Drive only.
# Written under a temporary name and then renamed, so two runtimes never leave a torn archive.
archive = f'{DRIVE_ROOT}/answers_revision4.zip'
tmp = f'{archive}.{uuid.uuid4().hex}.tmp'
n_files = 0
with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(ANSWERS_DIR):
        for fn in sorted(files):
            if (fn.endswith('.jsonl') or fn.endswith('__fit.json') or fn == 'DONE_revision4'
                    or fn == 'ALL_DONE_revision4'
                    or (fn.startswith('run_log_revision4') and fn.endswith('.json'))):
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, ANSWERS_DIR))
                n_files += 1
os.replace(tmp, archive)
print(f'hand-back archive: {archive} ({n_files} files)')
